In [4]:
# ============================================================
# PHASE 1.1 — IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [5]:
# ============================================================
# SYNTHETIC CHENNAI / ENNORE AIS DATASET
# For NEXFORGE / SPILLNET integration testing
# ============================================================

import numpy as np
import pandas as pd

np.random.seed(42)

# ------------------------------------------------------------
# 1. TARGET SOURCE FROM LAGRANGIAN MODEL
# ------------------------------------------------------------

SOURCE_LAT = 12.22076
SOURCE_LON = 80.13420
SOURCE_TIME = pd.Timestamp("2026-09-08 06:00:00", tz="UTC")

# ------------------------------------------------------------
# 2. DATASET SETTINGS
# ------------------------------------------------------------

NUM_VESSELS = 25
START_TIME = SOURCE_TIME - pd.Timedelta(hours=2)
END_TIME = SOURCE_TIME + pd.Timedelta(hours=2)

TIME_STEP = pd.Timedelta(minutes=5)

times = pd.date_range(
    START_TIME,
    END_TIME,
    freq=TIME_STEP,
    tz="UTC"
)

# ------------------------------------------------------------
# 3. VESSEL INFORMATION
# ------------------------------------------------------------

vessel_types = [
    70,  # Cargo
    80,  # Tanker
    60,  # Passenger
    30,  # Fishing
    90   # Other
]

vessel_names = [
    "CHENNAI TRADER",
    "EASTERN STAR",
    "BAY OF BENGAL",
    "COROMANDEL EXPRESS",
    "MARINE PRIDE",
    "OCEAN VOYAGER",
    "COASTAL SPIRIT",
    "SOUTHERN MERIDIAN",
    "INDIA STAR",
    "SEA HORIZON",
    "CORAL WAVE",
    "EAST COAST TRADER",
    "BLUE OCEAN",
    "CHENNAI MARINER",
    "BAY EXPRESS",
    "OCEAN GLORY",
    "COASTAL QUEEN",
    "MARINE STAR",
    "INDIAN VOYAGER",
    "EASTERN VOYAGER",
    "SEA BREEZE",
    "CHENNAI STAR",
    "COASTAL TRADER",
    "BAY MARINER",
    "OCEAN MERCHANT"
]

records = []

# ------------------------------------------------------------
# 4. GENERATE VESSEL TRACKS
# ------------------------------------------------------------

for vessel_id in range(NUM_VESSELS):

    mmsi = 419000000 + vessel_id

    vessel_type = np.random.choice(vessel_types)

    # --------------------------------------------------------
    # Give each vessel a different starting position
    # --------------------------------------------------------

    start_lat = SOURCE_LAT + np.random.uniform(-0.10, 0.10)
    start_lon = SOURCE_LON + np.random.uniform(-0.10, 0.10)

    # Different speed and course for each vessel
    speed_knots = np.random.uniform(4, 16)
    course = np.random.uniform(0, 360)

    # --------------------------------------------------------
    # Special vessel 0:
    # Make it pass close to the estimated source.
    # This acts as our controlled candidate vessel.
    # --------------------------------------------------------

    if vessel_id == 0:

        start_lat = SOURCE_LAT - 0.025
        start_lon = SOURCE_LON - 0.020

        course = 45
        speed_knots = 10

    lat = start_lat
    lon = start_lon

    for current_time in times:

        # ----------------------------------------------------
        # Convert speed from knots to km/h
        # ----------------------------------------------------

        speed_kmh = speed_knots * 1.852

        # Distance travelled during 5 minutes
        distance_km = speed_kmh * (5 / 60)

        # Convert course to radians
        course_rad = np.radians(course)

        # North/south movement
        delta_lat = (
            distance_km * np.cos(course_rad)
        ) / 111.32

        # East/west movement
        delta_lon = (
            distance_km * np.sin(course_rad)
        ) / (
            111.32 * np.cos(np.radians(lat))
        )

        lat += delta_lat
        lon += delta_lon

        records.append({
            "MMSI": mmsi,
            "BaseDateTime": current_time,
            "LAT": lat,
            "LON": lon,
            "SOG": speed_knots,
            "COG": course,
            "Heading": course,
            "VesselName": vessel_names[vessel_id],
            "IMO": 9000000 + vessel_id,
            "CallSign": f"IND{vessel_id:03d}",
            "VesselType": vessel_type,
            "Status": 0,
            "Length": np.random.uniform(80, 250),
            "Width": np.random.uniform(10, 40),
            "Draft": np.random.uniform(4, 12),
            "Cargo": 0,
            "TransceiverClass": "A"
        })

# ------------------------------------------------------------
# 5. CREATE DATAFRAME
# ------------------------------------------------------------

chennai_ais = pd.DataFrame(records)

# ------------------------------------------------------------
# 6. SORT
# ------------------------------------------------------------

chennai_ais = chennai_ais.sort_values(
    ["MMSI", "BaseDateTime"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 7. SAVE
# ------------------------------------------------------------

chennai_ais.to_csv(
    "synthetic_chennai_ais.csv",
    index=False
)

# ------------------------------------------------------------
# 8. BASIC INFORMATION
# ------------------------------------------------------------

print("==========================================")
print("SYNTHETIC CHENNAI AIS DATASET")
print("==========================================")

print(f"Rows           : {len(chennai_ais):,}")
print(f"Unique vessels : {chennai_ais['MMSI'].nunique()}")
print(f"Start time     : {chennai_ais['BaseDateTime'].min()}")
print(f"End time       : {chennai_ais['BaseDateTime'].max()}")

print("\nColumns:")
print(chennai_ais.columns.tolist())

print("\nFirst 5 rows:")
display(chennai_ais.head())

SYNTHETIC CHENNAI AIS DATASET
Rows           : 1,225
Unique vessels : 25
Start time     : 2026-09-08 04:00:00+00:00
End time       : 2026-09-08 08:00:00+00:00

Columns:
['MMSI', 'BaseDateTime', 'LAT', 'LON', 'SOG', 'COG', 'Heading', 'VesselName', 'IMO', 'CallSign', 'VesselType', 'Status', 'Length', 'Width', 'Draft', 'Cargo', 'TransceiverClass']

First 5 rows:


,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass
0,419000000,2026-09-08 04:00:00+00:00,12.205563,80.124230,10.0,45.0,45.0,CHENNAI TRADER,9000000,IND000,30,0,106.519068,11.742508,10.929409,0,A
1,419000000,2026-09-08 04:05:00+00:00,12.215367,80.134260,10.0,45.0,45.0,CHENNAI TRADER,9000000,IND000,30,0,182.189552,31.242177,4.164676,0,A
2,419000000,2026-09-08 04:10:00+00:00,12.225170,80.144290,10.0,45.0,45.0,CHENNAI TRADER,9000000,IND000,30,0,244.884675,34.973279,5.698713,0,A
3,419000000,2026-09-08 04:15:00+00:00,12.234973,80.154321,10.0,45.0,45.0,CHENNAI TRADER,9000000,IND000,30,0,110.910244,15.502135,6.433938,0,A
4,419000000,2026-09-08 04:20:00+00:00,12.244776,80.164352,10.0,45.0,45.0,CHENNAI TRADER,9000000,IND000,30,0,169.208593,22.958351,6.329833,0,A


In [8]:
# ============================================================
# PHASE 1.2 — LOAD AIS DATA
# ============================================================

from google.colab import files

uploaded = files.upload()

Saving AIS_178896334879276928_691-1788963349118.csv to AIS_178896334879276928_691-1788963349118.csv


In [9]:
# ============================================================
# PHASE 1.3 — READ AIS DATA
# ============================================================

file_name = list(uploaded.keys())[0]

ais = pd.read_csv(file_name)

print("AIS dataset loaded successfully.")
print("Rows    :", len(ais))
print("Columns :", len(ais.columns))

AIS dataset loaded successfully.
Rows    : 162637
Columns : 17


In [10]:
# ============================================================
# PHASE 1.4 — INSPECT AIS COLUMNS
# ============================================================

print("AIS columns:")
print(ais.columns.tolist())

AIS columns:
['MMSI', 'BaseDateTime', 'LAT', 'LON', 'SOG', 'COG', 'Heading', 'VesselName', 'IMO', 'CallSign', 'VesselType', 'Status', 'Length', 'Width', 'Draft', 'Cargo', 'TransceiverClass']


In [11]:
# ============================================================
# PHASE 1.5 — CHECK DATA TYPES
# ============================================================

print(ais.dtypes)

MMSI                  int64
BaseDateTime         object
LAT                 float64
LON                 float64
SOG                 float64
COG                 float64
Heading             float64
VesselName           object
IMO                  object
CallSign             object
VesselType          float64
Status              float64
Length              float64
Width               float64
Draft               float64
Cargo               float64
TransceiverClass     object
dtype: object


In [12]:
# ============================================================
# PHASE 1.5 — CONVERT AIS TIMESTAMP TO UTC
# ============================================================

ais["BaseDateTime"] = pd.to_datetime(
    ais["BaseDateTime"],
    utc=True
)

print("Timestamp converted to UTC.")
print()
print("Data type:")
print(ais["BaseDateTime"].dtype)
print()
print("First 5 timestamps:")
print(ais["BaseDateTime"].head())

Timestamp converted to UTC.

Data type:
datetime64[ns, UTC]

First 5 timestamps:
0   2022-01-01 00:00:00+00:00
1   2022-01-01 00:00:00+00:00
2   2022-01-01 00:00:02+00:00
3   2022-01-01 00:00:02+00:00
4   2022-01-01 00:00:51+00:00
Name: BaseDateTime, dtype: datetime64[ns, UTC]


In [13]:
# ============================================================
# PHASE 1.6 — CHECK AIS TIME RANGE
# ============================================================

print("Earliest AIS timestamp:")
print(ais["BaseDateTime"].min())

print()

print("Latest AIS timestamp:")
print(ais["BaseDateTime"].max())

Earliest AIS timestamp:
2021-12-31 00:00:00+00:00

Latest AIS timestamp:
2022-01-04 23:59:58+00:00


In [14]:
# ============================================================
# PHASE 1.7 — CHECK MISSING VALUES
# ============================================================

missing_values = ais.isnull().sum()

print("Missing values in each column:")
print(missing_values)

Missing values in each column:
MMSI                    0
BaseDateTime            0
LAT                     0
LON                     0
SOG                     0
COG                     0
Heading                 0
VesselName            285
IMO                 41556
CallSign              376
VesselType            700
Status               1601
Length               2885
Width               24562
Draft               54429
Cargo                1668
TransceiverClass        0
dtype: int64


In [15]:
# ============================================================
# PHASE 1.8 — CLEAN AND SORT AIS DATA
# ============================================================

# Make sure important numeric columns are numeric
numeric_columns = [
    "LAT",
    "LON",
    "SOG",
    "COG",
    "Heading",
    "VesselType",
    "Status",
    "Length",
    "Width",
    "Draft",
    "Cargo"
]

for column in numeric_columns:
    ais[column] = pd.to_numeric(ais[column], errors="coerce")

# Remove rows only if essential navigation data is missing
ais = ais.dropna(
    subset=["MMSI", "BaseDateTime", "LAT", "LON", "SOG", "COG"]
)

# Sort by vessel and time
ais = ais.sort_values(
    by=["MMSI", "BaseDateTime"]
).reset_index(drop=True)

print("AIS data cleaned and sorted successfully.")
print()
print("Rows after cleaning:", len(ais))
print("Unique vessels:", ais["MMSI"].nunique())

AIS data cleaned and sorted successfully.

Rows after cleaning: 162637
Unique vessels: 318


In [16]:
# ============================================================
# PHASE 1.9 — VERIFY CLEANED AIS DATA
# ============================================================

print("Dataset shape:")
print(ais.shape)

print()
print("First 5 records:")
print(ais.head())

print()
print("Last 5 records:")
print(ais.tail())

print()
print("Unique MMSIs:", ais["MMSI"].nunique())

Dataset shape:
(162637, 17)

First 5 records:
      MMSI              BaseDateTime       LAT       LON  SOG    COG  Heading  \
0  3660524 2021-12-31 01:57:26+00:00  28.15981 -89.23908  0.0  245.3    511.0   
1  3660524 2021-12-31 03:39:29+00:00  28.15980 -89.23904  0.1  258.0    511.0   
2  3660524 2021-12-31 03:51:21+00:00  28.15985 -89.23890  0.1  253.9    511.0   
3  3660524 2021-12-31 03:54:30+00:00  28.15978 -89.23909  0.0  253.9    511.0   
4  3660524 2021-12-31 04:09:29+00:00  28.15984 -89.23899  0.1  253.9    511.0   

    VesselName  IMO CallSign  VesselType  Status  Length  Width  Draft  Cargo  \
0  OLYMPUS TLP  NaN  WQOC607        90.0     1.0   110.0    NaN    NaN   80.0   
1  OLYMPUS TLP  NaN  WQOC607        90.0     1.0   110.0    NaN    NaN   80.0   
2  OLYMPUS TLP  NaN  WQOC607        90.0     1.0   110.0    NaN    NaN   80.0   
3  OLYMPUS TLP  NaN  WQOC607        90.0     1.0   110.0    NaN    NaN   80.0   
4  OLYMPUS TLP  NaN  WQOC607        90.0     1.0   110.0    Na

In [ ]:
# ============================================================
# PHASE 1.10 — AIS DATASET SUMMARY
# ============================================================

print("========== AIS DATASET SUMMARY ==========")

print("Total AIS records :", len(ais))
print("Unique vessels    :", ais["MMSI"].nunique())

print()
print("Time range:")
print("Start:", ais["BaseDateTime"].min())
print("End  :", ais["BaseDateTime"].max())

print()
print("Geographic range:")
print("Minimum latitude :", ais["LAT"].min())
print("Maximum latitude :", ais["LAT"].max())
print("Minimum longitude:", ais["LON"].min())
print("Maximum longitude:", ais["LON"].max())

print()
print("Speed range:")
print("Minimum SOG:", ais["SOG"].min(), "knots")
print("Maximum SOG:", ais["SOG"].max(), "knots")
print("Average SOG:", round(ais["SOG"].mean(), 2), "knots")

In [17]:
# ============================================================
# PHASE 2.1 — CALCULATE AIS TIME GAPS
# ============================================================

# Calculate time difference between consecutive AIS points
# for each vessel.

ais["time_diff_seconds"] = (
    ais.groupby("MMSI")["BaseDateTime"]
    .diff()
    .dt.total_seconds()
)

print("Time-gap calculation completed.")
print()

print("First 10 time gaps:")
print(
    ais[
        ["MMSI", "BaseDateTime", "time_diff_seconds"]
    ].head(10)
)

Time-gap calculation completed.

First 10 time gaps:
      MMSI              BaseDateTime  time_diff_seconds
0  3660524 2021-12-31 01:57:26+00:00                NaN
1  3660524 2021-12-31 03:39:29+00:00             6123.0
2  3660524 2021-12-31 03:51:21+00:00              712.0
3  3660524 2021-12-31 03:54:30+00:00              189.0
4  3660524 2021-12-31 04:09:29+00:00              899.0
5  3660524 2021-12-31 04:36:28+00:00             1619.0
6  3660524 2021-12-31 05:09:27+00:00             1979.0
7  3660524 2021-12-31 05:45:25+00:00             2158.0
8  3660524 2021-12-31 05:51:26+00:00              361.0
9  3660524 2021-12-31 06:00:27+00:00              541.0


In [18]:
# ============================================================
# PHASE 2.2 — CALCULATE DISTANCE BETWEEN AIS POINTS
# ============================================================

def haversine_distance_km(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance between two
    geographic coordinates in kilometers.
    """

    R = 6371.0  # Earth

In [19]:
# ============================================================
# PHASE 2.2 — CALCULATE DISTANCE BETWEEN AIS POINTS
# ============================================================

def haversine_distance_km(lat1, lon1, lat2, lon2):
    """
    Calculate great-circle distance between two
    geographic coordinates in kilometers.
    """

    R = 6371.0  # Earth radius in km

    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)

    dlat = lat2 - lat1
    dlon = np.radians(lon2) - np.radians(lon1)

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c


# Previous latitude and longitude for each vessel
ais["previous_LAT"] = (
    ais.groupby("MMSI")["LAT"].shift(1)
)

ais["previous_LON"] = (
    ais.groupby("MMSI")["LON"].shift(1)
)


# Calculate distance from previous AIS point
ais["distance_from_previous_km"] = haversine_distance_km(
    ais["previous_LAT"],
    ais["previous_LON"],
    ais["LAT"],
    ais["LON"]
)


# First point of every vessel has no previous point
ais["distance_from_previous_km"] = (
    ais["distance_from_previous_km"].fillna(0)
)


print("Distance calculation completed.")
print()

print(
    ais[
        [
            "MMSI",
            "LAT",
            "LON",
            "previous_LAT",
            "previous_LON",
            "distance_from_previous_km"
        ]
    ].head(10)
)

Distance calculation completed.

      MMSI       LAT       LON  previous_LAT  previous_LON  \
0  3660524  28.15981 -89.23908           NaN           NaN   
1  3660524  28.15980 -89.23904      28.15981     -89.23908   
2  3660524  28.15985 -89.23890      28.15980     -89.23904   
3  3660524  28.15978 -89.23909      28.15985     -89.23890   
4  3660524  28.15984 -89.23899      28.15978     -89.23909   
5  3660524  28.15984 -89.23911      28.15984     -89.23899   
6  3660524  28.15990 -89.23896      28.15984     -89.23911   
7  3660524  28.15984 -89.23907      28.15990     -89.23896   
8  3660524  28.15982 -89.23905      28.15984     -89.23907   
9  3660524  28.15981 -89.23909      28.15982     -89.23905   

   distance_from_previous_km  
0                   0.000000  
1                   0.004076  
2                   0.014808  
3                   0.020187  
4                   0.011858  
5                   0.011764  
6                   0.016148  
7                   0.012681  
8    

In [20]:
# ============================================================
# PHASE 2.3 — CALCULATE MOVEMENT SPEED
# ============================================================

# Calculate speed from distance and time.
# Distance is in km.
# Time is converted from seconds to hours.
# Resulting speed is km/h.

ais["calculated_speed_kmh"] = np.where(
    ais["time_diff_seconds"] > 0,
    ais["distance_from_previous_km"]
    / (ais["time_diff_seconds"] / 3600),
    0
)

# Convert km/h to knots
ais["calculated_speed_knots"] = (
    ais["calculated_speed_kmh"] / 1.852
)

print("Calculated movement speed successfully.")
print()

print(
    ais[
        [
            "MMSI",
            "BaseDateTime",
            "SOG",
            "distance_from_previous_km",
            "time_diff_seconds",
            "calculated_speed_knots"
        ]
    ].head(10)
)

Calculated movement speed successfully.

      MMSI              BaseDateTime  SOG  distance_from_previous_km  \
0  3660524 2021-12-31 01:57:26+00:00  0.0                   0.000000   
1  3660524 2021-12-31 03:39:29+00:00  0.1                   0.004076   
2  3660524 2021-12-31 03:51:21+00:00  0.1                   0.014808   
3  3660524 2021-12-31 03:54:30+00:00  0.0                   0.020187   
4  3660524 2021-12-31 04:09:29+00:00  0.1                   0.011858   
5  3660524 2021-12-31 04:36:28+00:00  0.1                   0.011764   
6  3660524 2021-12-31 05:09:27+00:00  0.1                   0.016148   
7  3660524 2021-12-31 05:45:25+00:00  0.1                   0.012681   
8  3660524 2021-12-31 05:51:26+00:00  0.0                   0.002965   
9  3660524 2021-12-31 06:00:27+00:00  0.1                   0.004076   

   time_diff_seconds  calculated_speed_knots  
0                NaN                0.000000  
1             6123.0                0.001294  
2              712.0     

In [21]:
# ============================================================
# PHASE 2.4 — COMPARE REPORTED AND CALCULATED SPEED
# ============================================================

ais["speed_difference_knots"] = (
    ais["SOG"] - ais["calculated_speed_knots"]
)

print("Speed comparison calculated successfully.")
print()

print(
    ais[
        [
            "MMSI",
            "BaseDateTime",
            "SOG",
            "calculated_speed_knots",
            "speed_difference_knots"
        ]
    ].head(10)
)

print()
print("Average reported SOG:",
      round(ais["SOG"].mean(), 2), "knots")

print("Average calculated speed:",
      round(ais["calculated_speed_knots"].mean(), 2), "knots")

Speed comparison calculated successfully.

      MMSI              BaseDateTime  SOG  calculated_speed_knots  \
0  3660524 2021-12-31 01:57:26+00:00  0.0                0.000000   
1  3660524 2021-12-31 03:39:29+00:00  0.1                0.001294   
2  3660524 2021-12-31 03:51:21+00:00  0.1                0.040428   
3  3660524 2021-12-31 03:54:30+00:00  0.0                0.207624   
4  3660524 2021-12-31 04:09:29+00:00  0.1                0.025640   
5  3660524 2021-12-31 04:36:28+00:00  0.1                0.014124   
6  3660524 2021-12-31 05:09:27+00:00  0.1                0.015861   
7  3660524 2021-12-31 05:45:25+00:00  0.1                0.011422   
8  3660524 2021-12-31 05:51:26+00:00  0.0                0.015964   
9  3660524 2021-12-31 06:00:27+00:00  0.1                0.014645   

   speed_difference_knots  
0                0.000000  
1                0.098706  
2                0.059572  
3               -0.207624  
4                0.074360  
5                0.085876  
6

In [22]:
# ============================================================
# PHASE 2.5 — IDENTIFY LONG AIS GAPS
# ============================================================

# Prototype threshold for a long AIS gap
AIS_GAP_THRESHOLD_MINUTES = 30

ais["ais_gap_minutes"] = (
    ais["time_diff_seconds"] / 60
)

ais["is_long_ais_gap"] = (
    ais["ais_gap_minutes"] > AIS_GAP_THRESHOLD_MINUTES
)

print("AIS gap analysis completed.")
print()

print(
    "AIS gap threshold:",
    AIS_GAP_THRESHOLD_MINUTES,
    "minutes"
)

print()

print(
    "Number of long AIS gaps:",
    ais["is_long_ais_gap"].sum()
)

print()

print(
    "Percentage of AIS records after long gaps:",
    round(
        ais["is_long_ais_gap"].mean() * 100,
        2
    ),
    "%"
)

AIS gap analysis completed.

AIS gap threshold: 30 minutes

Number of long AIS gaps: 540

Percentage of AIS records after long gaps: 0.33 %


In [23]:
# ============================================================
# PHASE 2.6 — CALCULATE COURSE CHANGE
# ============================================================

# Previous course for each vessel
ais["previous_COG"] = (
    ais.groupby("MMSI")["COG"].shift(1)
)

# Calculate the smallest angular difference
ais["course_change_deg"] = (
    (
        ais["COG"]
        - ais["previous_COG"]
        + 180
    ) % 360
) - 180

# First point of each vessel has no previous course
ais["course_change_deg"] = (
    ais["course_change_deg"].fillna(0)
)

print("Course-change calculation completed.")
print()

print(
    ais[
        [
            "MMSI",
            "BaseDateTime",
            "COG",
            "previous_COG",
            "course_change_deg"
        ]
    ].head(10)
)

Course-change calculation completed.

      MMSI              BaseDateTime    COG  previous_COG  course_change_deg
0  3660524 2021-12-31 01:57:26+00:00  245.3           NaN                0.0
1  3660524 2021-12-31 03:39:29+00:00  258.0         245.3               12.7
2  3660524 2021-12-31 03:51:21+00:00  253.9         258.0               -4.1
3  3660524 2021-12-31 03:54:30+00:00  253.9         253.9                0.0
4  3660524 2021-12-31 04:09:29+00:00  253.9         253.9                0.0
5  3660524 2021-12-31 04:36:28+00:00  253.9         253.9                0.0
6  3660524 2021-12-31 05:09:27+00:00  253.9         253.9                0.0
7  3660524 2021-12-31 05:45:25+00:00  253.9         253.9                0.0
8  3660524 2021-12-31 05:51:26+00:00  253.9         253.9                0.0
9  3660524 2021-12-31 06:00:27+00:00  253.9         253.9                0.0


In [24]:
# ============================================================
# PHASE 2.7 — CLEAN INVALID AIS HEADING VALUES
# ============================================================

# AIS uses 511 to indicate that heading is unavailable.
# Replace it with NaN.

invalid_heading_count = (
    (ais["Heading"] == 511).sum()
)

ais.loc[ais["Heading"] == 511, "Heading"] = np.nan

print("Invalid heading values found:", invalid_heading_count)
print("Invalid heading values replaced with NaN.")

print()
print("Remaining valid heading values:")
print(ais["Heading"].notna().sum())

Invalid heading values found: 55174
Invalid heading values replaced with NaN.

Remaining valid heading values:
107463


In [25]:
# ============================================================
# PHASE 2.8 — CREATE VESSEL-LEVEL TRACK SUMMARY
# ============================================================

vessel_summary = (
    ais.groupby("MMSI")
    .agg(
        point_count=("MMSI", "size"),
        start_time=("BaseDateTime", "min"),
        end_time=("BaseDateTime", "max"),
        average_sog=("SOG", "mean"),
        maximum_sog=("SOG", "max"),
        total_distance_km=("distance_from_previous_km", "sum"),
        average_course_change=("course_change_deg", "mean"),
        long_ais_gaps=("is_long_ais_gap", "sum")
    )
    .reset_index()
)

print("Vessel-level summary created.")
print()

print("Number of vessels:", len(vessel_summary))

print()
print(vessel_summary.head(10))

Vessel-level summary created.

Number of vessels: 318

        MMSI  point_count                start_time                  end_time  \
0    3660524          164 2021-12-31 01:57:26+00:00 2022-01-02 04:06:39+00:00   
1  205125000          186 2022-01-02 11:46:40+00:00 2022-01-02 15:30:10+00:00   
2  209388000           58 2022-01-01 20:02:15+00:00 2022-01-02 01:35:58+00:00   
3  209677000          121 2022-01-04 09:32:17+00:00 2022-01-04 13:17:46+00:00   
4  209716000          165 2021-12-31 00:09:57+00:00 2021-12-31 03:18:09+00:00   
5  210347000          259 2022-01-01 04:44:01+00:00 2022-01-01 10:15:30+00:00   
6  212049000          110 2022-01-04 06:59:18+00:00 2022-01-04 10:44:07+00:00   
7  212370000          105 2022-01-04 05:35:02+00:00 2022-01-04 09:38:44+00:00   
8  215031000          290 2021-12-31 20:53:04+00:00 2022-01-01 02:47:54+00:00   
9  218629000          767 2022-01-01 21:44:59+00:00 2022-01-03 12:02:07+00:00   

   average_sog  maximum_sog  total_distance_km  avera

In [26]:
# ============================================================
# PHASE 3.1 — CREATE CONTROLLED SOURCE EVENT
# ============================================================

SOURCE_MMSI = 205125000

source_track = ais[
    ais["MMSI"] == SOURCE_MMSI
].copy()

# Select a middle point from the vessel's trajectory
source_index = len(source_track) // 2

source_event = source_track.iloc[source_index]

spill_lat = source_event["LAT"]
spill_lon = source_event["LON"]
spill_time = source_event["BaseDateTime"]

print("Controlled source event created.")
print()

print("Source vessel MMSI :", SOURCE_MMSI)
print("Spill latitude     :", spill_lat)
print("Spill longitude    :", spill_lon)
print("Spill time         :", spill_time)

Controlled source event created.

Source vessel MMSI : 205125000
Spill latitude     : 28.69912
Spill longitude    : -89.35419
Spill time         : 2022-01-02 13:42:11+00:00


In [27]:
# ============================================================
# PHASE 3.2 — FIND SPATIAL + TEMPORAL CANDIDATES
# ============================================================

SEARCH_RADIUS_KM = 5.0
TIME_WINDOW_MINUTES = 30.0

# Calculate distance from every AIS point to the
# controlled spill origin.

ais["distance_to_origin_km"] = haversine_distance_km(
    spill_lat,
    spill_lon,
    ais["LAT"],
    ais["LON"]
)

# Calculate time difference from the controlled event.

ais["time_difference_minutes"] = (
    (
        ais["BaseDateTime"] - spill_time
    ).abs().dt.total_seconds() / 60
)

# Apply spatial and temporal filters.

candidate_points = ais[
    (ais["distance_to_origin_km"] <= SEARCH_RADIUS_KM) &
    (ais["time_difference_minutes"] <= TIME_WINDOW_MINUTES)
].copy()

print("Candidate search completed.")
print()

print("Search radius:", SEARCH_RADIUS_KM, "km")
print("Time window  :", TIME_WINDOW_MINUTES, "minutes")

print()
print("Candidate AIS records:", len(candidate_points))

print(
    "Candidate vessels:",
    candidate_points["MMSI"].nunique()
)

print()
print("Candidate MMSIs:")
print(
    candidate_points["MMSI"]
    .unique()
)

Candidate search completed.

Search radius: 5.0 km
Time window  : 30.0 minutes

Candidate AIS records: 80
Candidate vessels: 3

Candidate MMSIs:
[205125000 368067520 563085200]


In [28]:
# ============================================================
# PHASE 3.3 — BUILD CANDIDATE-LEVEL FEATURES
# ============================================================

candidate_features = (
    candidate_points
    .groupby("MMSI")
    .agg(
        closest_distance_km=(
            "distance_to_origin_km",
            "min"
        ),

        closest_time_difference_min=(
            "time_difference_minutes",
            "min"
        ),

        average_sog=(
            "SOG",
            "mean"
        ),

        maximum_sog=(
            "SOG",
            "max"
        ),

        ais_point_count=(
            "MMSI",
            "size"
        ),

        average_course_change=(
            "course_change_deg",
            lambda x: x.abs().mean()
        )
    )
    .reset_index()
)

print("Candidate-level features created.")
print()

print(candidate_features)

Candidate-level features created.

        MMSI  closest_distance_km  closest_time_difference_min  average_sog  \
0  205125000             0.000000                     0.000000    12.845000   
1  368067520             4.083712                     0.366667     0.000000   
2  563085200             1.514082                     0.083333    13.338889   

   maximum_sog  ais_point_count  average_course_change  
0         13.1               20               2.775000  
1          0.0               42               5.023810  
2         13.8               18               1.161111  


In [29]:
import numpy as np

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [30]:
import numpy as np
import pandas as pd


# ============================================================
# PHASE 3.4 — FINAL FEATURE ENGINEERING
# ============================================================

def calculate_candidate_features(
    vessel_df,
    source_lat,
    source_lon,
    source_time,
    source_radius_km=5
):
    """
    Calculate the final AIS features for one candidate vessel
    around an estimated oil-spill source event.
    """

    df = vessel_df.copy()

    # --------------------------------------------------------
    # 1. Make sure timestamp is datetime
    # --------------------------------------------------------
    df["BaseDateTime"] = pd.to_datetime(
        df["BaseDateTime"],
        utc=True
    )

    source_time = pd.to_datetime(
        source_time,
        utc=True
    )

    # --------------------------------------------------------
    # 2. Calculate distance from estimated spill origin
    # --------------------------------------------------------
    df["distance_to_source_km"] = haversine_distance(
        source_lat,
        source_lon,
        df["LAT"].values,
        df["LON"].values
    )

    # --------------------------------------------------------
    # 3. Time difference from estimated release time
    # --------------------------------------------------------
    df["time_difference_min"] = (
        (df["BaseDateTime"] - source_time)
        .abs()
        .dt.total_seconds()
        / 60
    )

    # --------------------------------------------------------
    # 4. Closest point to source
    # --------------------------------------------------------
    closest_idx = df["distance_to_source_km"].idxmin()

    closest_row = df.loc[closest_idx]

    closest_distance = closest_row["distance_to_source_km"]

    closest_time_difference = closest_row["time_difference_min"]

    speed_at_closest = closest_row["SOG"]

    course_change_at_closest = closest_row.get(
        "course_change_deg",
        np.nan
    )

    # --------------------------------------------------------
    # 5. Distance at beginning and end of candidate window
    # --------------------------------------------------------
    first_distance = df.iloc[0]["distance_to_source_km"]
    last_distance = df.iloc[-1]["distance_to_source_km"]

    distance_change = first_distance - last_distance

    # Positive value:
    # vessel ended closer to source than it started
    #
    # Negative value:
    # vessel ended farther from source than it started

    # --------------------------------------------------------
    # 6. Time spent inside source radius
    # --------------------------------------------------------
    inside_radius = df[
        df["distance_to_source_km"] <= source_radius_km
    ]

    if len(inside_radius) >= 2:

        time_inside_radius = (
            inside_radius["BaseDateTime"].max()
            - inside_radius["BaseDateTime"].min()
        ).total_seconds() / 60

    else:
        time_inside_radius = 0.0

    # --------------------------------------------------------
    # 7. AIS point count
    # --------------------------------------------------------
    ais_point_count = len(df)

    # --------------------------------------------------------
    # 8. Speed features
    # --------------------------------------------------------
    average_sog = df["SOG"].mean()

    maximum_sog = df["SOG"].max()

    speed_change = (
        df.iloc[-1]["SOG"]
        - df.iloc[0]["SOG"]
    )

    # --------------------------------------------------------
    # 9. Course-change features
    # --------------------------------------------------------
    if "course_change_deg" in df.columns:

        average_course_change = (
            df["course_change_deg"]
            .dropna()
            .mean()
        )

    else:

        average_course_change = np.nan

    # --------------------------------------------------------
    # 10. Approach speed
    # --------------------------------------------------------
    # Use observations before the closest point

    before_closest = df[
        df["BaseDateTime"] <= closest_row["BaseDateTime"]
    ]

    if len(before_closest) > 0:
        approach_speed = before_closest["SOG"].mean()
    else:
        approach_speed = 0.0

    # --------------------------------------------------------
    # 11. Departure speed
    # --------------------------------------------------------
    # Use observations after the closest point

    after_closest = df[
        df["BaseDateTime"] >= closest_row["BaseDateTime"]
    ]

    if len(after_closest) > 0:
        departure_speed = after_closest["SOG"].mean()
    else:
        departure_speed = 0.0

    # --------------------------------------------------------
    # 12. AIS gap features
    # --------------------------------------------------------
    if "time_diff_seconds" in df.columns:

        maximum_ais_gap_min = (
            df["time_diff_seconds"]
            .max()
            / 60
        )

        long_ais_gap_count = (
            df["is_long_ais_gap"]
            .sum()
            if "is_long_ais_gap" in df.columns
            else 0
        )

    else:

        time_diffs = (
            df["BaseDateTime"]
            .diff()
            .dt.total_seconds()
        )

        maximum_ais_gap_min = (
            time_diffs.max() / 60
        )

        long_ais_gap_count = (
            (time_diffs > 30 * 60).sum()
        )

    # --------------------------------------------------------
    # 13. Moving fraction
    # --------------------------------------------------------
    moving_fraction = (
        (df["SOG"] > 0.5).mean()
    )

    # --------------------------------------------------------
    # 14. Return final feature dictionary
    # --------------------------------------------------------
    features = {

        "closest_distance_km":
            closest_distance,

        "distance_at_window_start_km":
            first_distance,

        "distance_at_window_end_km":
            last_distance,

        "distance_change_km":
            distance_change,

        "closest_time_difference_min":
            closest_time_difference,

        "time_inside_source_radius_min":
            time_inside_radius,

        "ais_point_count":
            ais_point_count,

        "average_sog":
            average_sog,

        "maximum_sog":
            maximum_sog,

        "speed_at_closest_approach":
            speed_at_closest,

        "speed_change":
            speed_change,

        "average_course_change":
            average_course_change,

        "course_change_at_closest_approach":
            course_change_at_closest,

        "approach_speed":
            approach_speed,

        "departure_speed":
            departure_speed,

        "maximum_ais_gap_min":
            maximum_ais_gap_min,

        "long_ais_gap_count":
            long_ais_gap_count,

        "moving_fraction":
            moving_fraction
    }

    return features

In [31]:
# Show all DataFrames currently available in Colab

dataframes = []

for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame):
        dataframes.append((name, value.shape))

for name, shape in dataframes:
    print(name, "→", shape)

chennai_ais → (1225, 17)
ais_chennai → (1225, 21)
ais → (162637, 30)
vessel_summary → (318, 9)
source_track → (186, 28)
candidate_points → (80, 30)
candidate_features → (3, 7)


In [32]:
# Show the DataFrames currently available in Colab

for name, value in globals().items():
    if isinstance(value, pd.DataFrame):
        print(name, "→", value.shape)

chennai_ais → (1225, 17)
ais_chennai → (1225, 21)
ais → (162637, 30)
vessel_summary → (318, 9)
source_track → (186, 28)
candidate_points → (80, 30)
candidate_features → (3, 7)


In [33]:
# ============================================================
# PHASE 3.4 — CREATE FINAL FEATURE TABLE
# ============================================================

# Controlled hypothetical spill source
source_lat = 28.69912
source_lon = -89.35419
source_time = pd.Timestamp("2022-01-02 13:42:11", tz="UTC")

# Candidate vessels from our previous filtering
candidate_mmsis = [
    205125000,
    563085200,
    368067520
]

# Candidate time window: ±30 minutes
time_window_minutes = 30

candidate_feature_rows = []


for mmsi in candidate_mmsis:

    # --------------------------------------------------------
    # Get this vessel's AIS records
    # --------------------------------------------------------
    vessel_df = ais[
        ais["MMSI"] == mmsi
    ].copy()

    # --------------------------------------------------------
    # Keep only records within ±30 minutes of source time
    # --------------------------------------------------------
    time_difference = (
        vessel_df["BaseDateTime"] - source_time
    ).abs()

    vessel_df = vessel_df[
        time_difference <= pd.Timedelta(
            minutes=time_window_minutes
        )
    ].copy()

    # Sort chronologically
    vessel_df = vessel_df.sort_values(
        "BaseDateTime"
    ).reset_index(drop=True)

    # Skip if no data
    if len(vessel_df) == 0:
        print(f"No data found for MMSI {mmsi}")
        continue

    # --------------------------------------------------------
    # Calculate our 18 final features
    # --------------------------------------------------------
    features = calculate_candidate_features(
        vessel_df=vessel_df,
        source_lat=source_lat,
        source_lon=source_lon,
        source_time=source_time,
        source_radius_km=5
    )

    # Add vessel identifier
    features["MMSI"] = mmsi

    candidate_feature_rows.append(features)


# ============================================================
# Convert results into DataFrame
# ============================================================

candidate_feature_df = pd.DataFrame(
    candidate_feature_rows
)


# Put MMSI as the first column
candidate_feature_df = candidate_feature_df[
    ["MMSI"] +
    [
        col
        for col in candidate_feature_df.columns
        if col != "MMSI"
    ]
]


# ============================================================
# Display
# ============================================================

print("FINAL CANDIDATE FEATURE TABLE")
print("=" * 70)

print(
    "Shape:",
    candidate_feature_df.shape
)

display(candidate_feature_df)

FINAL CANDIDATE FEATURE TABLE
Shape: (3, 19)


,MMSI,closest_distance_km,distance_at_window_start_km,distance_at_window_end_km,distance_change_km,closest_time_difference_min,time_inside_source_radius_min,ais_point_count,average_sog,maximum_sog,speed_at_closest_approach,speed_change,average_course_change,course_change_at_closest_approach,approach_speed,departure_speed,maximum_ais_gap_min,long_ais_gap_count,moving_fraction
0,205125000,0.000000,10.568729,11.638407,-1.069678,0.00,23.666667,51,12.613725,13.1,13.1,-0.5,1.258824,-2.2,12.592308,12.653846,1.516667,0,1.0
1,563085200,1.514082,13.623822,10.414654,3.209168,4.95,21.350000,49,13.226531,13.8,13.4,1.1,-0.532653,-1.6,12.962069,13.600000,1.683333,0,1.0
2,368067520,4.083712,4.088577,4.085658,0.002919,5.80,59.000000,42,0.000000,0.0,0.0,0.0,-5.023810,0.0,0.000000,0.000000,2.666667,0,0.0


In [34]:
# ============================================================
# FIX COURSE-CHANGE FEATURES
# ============================================================

# Course change should represent turning magnitude,
# therefore convert it to absolute degrees.

ais["course_change_deg"] = ais["course_change_deg"].abs()

print("Course-change feature corrected.")
print(
    ais["course_change_deg"]
    .describe()
)

Course-change feature corrected.
count    162637.000000
mean         16.197183
std          36.184594
min           0.000000
25%           0.200000
50%           1.700000
75%           9.000000
max         180.000000
Name: course_change_deg, dtype: float64


In [35]:
# ============================================================
# PHASE 3.4 — CREATE FINAL FEATURE TABLE
# ============================================================

source_lat = 28.69912
source_lon = -89.35419
source_time = pd.Timestamp("2022-01-02 13:42:11", tz="UTC")

candidate_mmsis = [
    205125000,
    563085200,
    368067520
]

time_window_minutes = 30

candidate_feature_rows = []

for mmsi in candidate_mmsis:

    vessel_df = ais[
        ais["MMSI"] == mmsi
    ].copy()

    time_difference = (
        vessel_df["BaseDateTime"] - source_time
    ).abs()

    vessel_df = vessel_df[
        time_difference <= pd.Timedelta(
            minutes=time_window_minutes
        )
    ].copy()

    vessel_df = vessel_df.sort_values(
        "BaseDateTime"
    ).reset_index(drop=True)

    if len(vessel_df) == 0:
        print(f"No data found for MMSI {mmsi}")
        continue

    features = calculate_candidate_features(
        vessel_df=vessel_df,
        source_lat=source_lat,
        source_lon=source_lon,
        source_time=source_time,
        source_radius_km=5
    )

    features["MMSI"] = mmsi

    candidate_feature_rows.append(features)


candidate_feature_df = pd.DataFrame(
    candidate_feature_rows
)

candidate_feature_df = candidate_feature_df[
    ["MMSI"] +
    [
        col
        for col in candidate_feature_df.columns
        if col != "MMSI"
    ]
]

print("FINAL CANDIDATE FEATURE TABLE")
print("=" * 70)

print("Shape:", candidate_feature_df.shape)

display(candidate_feature_df)

FINAL CANDIDATE FEATURE TABLE
Shape: (3, 19)


,MMSI,closest_distance_km,distance_at_window_start_km,distance_at_window_end_km,distance_change_km,closest_time_difference_min,time_inside_source_radius_min,ais_point_count,average_sog,maximum_sog,speed_at_closest_approach,speed_change,average_course_change,course_change_at_closest_approach,approach_speed,departure_speed,maximum_ais_gap_min,long_ais_gap_count,moving_fraction
0,205125000,0.000000,10.568729,11.638407,-1.069678,0.00,23.666667,51,12.613725,13.1,13.1,-0.5,4.800000,2.2,12.592308,12.653846,1.516667,0,1.0
1,563085200,1.514082,13.623822,10.414654,3.209168,4.95,21.350000,49,13.226531,13.8,13.4,1.1,3.014286,1.6,12.962069,13.600000,1.683333,0,1.0
2,368067520,4.083712,4.088577,4.085658,0.002919,5.80,59.000000,42,0.000000,0.0,0.0,0.0,5.023810,0.0,0.000000,0.000000,2.666667,0,0.0


In [36]:
# ============================================================
# PHASE 4.1 — GENERATE CONTROLLED SOURCE EVENTS
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

SOURCE_RADIUS_KM = 5
TIME_WINDOW_MINUTES = 30

NUMBER_OF_EVENTS = 100

RANDOM_STATE = 42


# ------------------------------------------------------------
# Make sure timestamps are datetime
# ------------------------------------------------------------

ais["BaseDateTime"] = pd.to_datetime(
    ais["BaseDateTime"],
    utc=True
)


# ------------------------------------------------------------
# Select realistic source points
#
# We use moving vessels only.
# A stationary vessel is not a good source-event example.
# ------------------------------------------------------------

source_pool = ais[
    (ais["SOG"] >= 2.0)
].copy()


print("Total AIS records:", len(ais))
print("Possible moving source points:", len(source_pool))


# ------------------------------------------------------------
# Randomly select source points
# ------------------------------------------------------------

source_events = source_pool.sample(
    n=min(NUMBER_OF_EVENTS, len(source_pool)),
    random_state=RANDOM_STATE
).reset_index(drop=True)


# ------------------------------------------------------------
# Create event table
# ------------------------------------------------------------

source_event_df = pd.DataFrame({
    "event_id": [
        f"EVENT_{i+1:04d}"
        for i in range(len(source_events))
    ],

    "source_mmsi": source_events["MMSI"].values,

    "source_lat": source_events["LAT"].values,

    "source_lon": source_events["LON"].values,

    "source_time": source_events["BaseDateTime"].values
})


print()
print("Generated controlled source events:")
print("=" * 60)

print("Number of events:", len(source_event_df))

display(source_event_df.head(10))

Total AIS records: 162637
Possible moving source points: 75551

Generated controlled source events:
Number of events: 100


,event_id,source_mmsi,source_lat,source_lon,source_time
0,EVENT_0001,477288000,28.69323,-89.55078,2022-01-03 08:34:51
1,EVENT_0002,369260000,28.77401,-89.66129,2022-01-02 13:56:46
2,EVENT_0003,538008391,28.74151,-89.51769,2022-01-01 00:28:10
3,EVENT_0004,538009555,28.55262,-89.21176,2022-01-04 19:41:13
4,EVENT_0005,311050800,28.94085,-89.40408,2022-01-02 03:28:57
5,EVENT_0006,367582190,28.85000,-89.47514,2022-01-04 22:57:02
6,EVENT_0007,368196410,28.69477,-89.99087,2022-01-02 01:24:34
7,EVENT_0008,367458710,28.85536,-89.38865,2022-01-02 10:15:43
8,EVENT_0009,367590950,28.90249,-89.52328,2022-01-04 23:32:30
9,EVENT_0010,538003726,28.77368,-89.46799,2022-01-01 01:10:28


In [37]:
# ============================================================
# PHASE 4.2 — FIND CANDIDATES + CREATE LABELS
# ============================================================

SOURCE_RADIUS_KM = 5
TIME_WINDOW_MINUTES = 30

training_candidate_rows = []


# ------------------------------------------------------------
# Process every controlled source event
# ------------------------------------------------------------

for _, event in source_event_df.iterrows():

    event_id = event["event_id"]

    source_mmsi = event["source_mmsi"]

    source_lat = event["source_lat"]
    source_lon = event["source_lon"]

    source_time = pd.to_datetime(
        event["source_time"],
        utc=True
    )

    # --------------------------------------------------------
    # Temporal filtering
    # --------------------------------------------------------

    time_difference = (
        ais["BaseDateTime"] - source_time
    ).abs()

    temporal_candidates = ais[
        time_difference <= pd.Timedelta(
            minutes=TIME_WINDOW_MINUTES
        )
    ].copy()

    # --------------------------------------------------------
    # Spatial filtering
    # --------------------------------------------------------

    temporal_candidates["distance_to_source_km"] = (
        haversine_distance(
            source_lat,
            source_lon,
            temporal_candidates["LAT"].values,
            temporal_candidates["LON"].values
        )
    )

    candidate_points = temporal_candidates[
        temporal_candidates["distance_to_source_km"]
        <= SOURCE_RADIUS_KM
    ].copy()

    # --------------------------------------------------------
    # Get unique candidate vessels
    # --------------------------------------------------------

    candidate_mmsis = (
        candidate_points["MMSI"]
        .dropna()
        .unique()
    )

    # --------------------------------------------------------
    # Create one feature row per candidate vessel
    # --------------------------------------------------------

    for candidate_mmsi in candidate_mmsis:

        vessel_df = candidate_points[
            candidate_points["MMSI"] == candidate_mmsi
        ].copy()

        vessel_df = vessel_df.sort_values(
            "BaseDateTime"
        ).reset_index(drop=True)

        # Need enough AIS observations
        if len(vessel_df) < 2:
            continue

        # ----------------------------------------------------
        # Calculate our 18 features
        # ----------------------------------------------------

        features = calculate_candidate_features(
            vessel_df=vessel_df,
            source_lat=source_lat,
            source_lon=source_lon,
            source_time=source_time,
            source_radius_km=SOURCE_RADIUS_KM
        )

        # ----------------------------------------------------
        # Add event information
        # ----------------------------------------------------

        features["event_id"] = event_id

        features["MMSI"] = candidate_mmsi

        # ----------------------------------------------------
        # Label
        # ----------------------------------------------------

        if candidate_mmsi == source_mmsi:
            features["label"] = 1
        else:
            features["label"] = 0

        training_candidate_rows.append(features)


# ============================================================
# Create final training-candidate DataFrame
# ============================================================

training_candidate_df = pd.DataFrame(
    training_candidate_rows
)


# ------------------------------------------------------------
# Reorder columns
# ------------------------------------------------------------

feature_columns = [
    "closest_distance_km",
    "distance_at_window_start_km",
    "distance_at_window_end_km",
    "distance_change_km",

    "closest_time_difference_min",
    "time_inside_source_radius_min",
    "ais_point_count",

    "average_sog",
    "maximum_sog",
    "speed_at_closest_approach",
    "speed_change",

    "average_course_change",
    "course_change_at_closest_approach",
    "approach_speed",
    "departure_speed",

    "maximum_ais_gap_min",
    "long_ais_gap_count",
    "moving_fraction"
]


training_candidate_df = training_candidate_df[
    ["event_id", "MMSI"] +
    feature_columns +
    ["label"]
]


# ============================================================
# Display results
# ============================================================

print("TRAINING CANDIDATE DATASET")
print("=" * 70)

print(
    "Shape:",
    training_candidate_df.shape
)

print(
    "Unique events:",
    training_candidate_df["event_id"].nunique()
)

print(
    "Unique vessels:",
    training_candidate_df["MMSI"].nunique()
)

print("\nLabel distribution:")
print(
    training_candidate_df["label"]
    .value_counts()
)

display(
    training_candidate_df.head(10)
)

TRAINING CANDIDATE DATASET
Shape: (310, 21)
Unique events: 100
Unique vessels: 154

Label distribution:
label
0    210
1    100
Name: count, dtype: int64


,event_id,MMSI,closest_distance_km,distance_at_window_start_km,distance_at_window_end_km,distance_change_km,closest_time_difference_min,time_inside_source_radius_min,ais_point_count,average_sog,...,speed_at_closest_approach,speed_change,average_course_change,course_change_at_closest_approach,approach_speed,departure_speed,maximum_ais_gap_min,long_ais_gap_count,moving_fraction,label
0,EVENT_0001,367373000,3.512318,3.529057,3.522263,0.006793,26.883333,59.000000,49,0.002041,...,0.0,0.0,3.414286,0.0,0.002128,0.000000,1.516667,0,0.0,0
1,EVENT_0001,477288000,0.000000,4.980145,4.591124,0.389022,0.000000,22.700000,20,13.595000,...,13.8,-0.4,1.000000,0.0,13.745455,13.450000,1.333333,0,1.0,1
2,EVENT_0001,538007823,2.594918,4.882694,2.594918,2.287776,29.383333,5.833333,6,12.616667,...,12.6,0.0,0.583333,0.4,12.616667,12.600000,1.500000,0,1.0,0
3,EVENT_0001,538008071,4.816536,4.846485,4.959095,-0.112610,28.800000,4.816667,5,1.920000,...,1.5,1.2,27.160000,47.7,1.500000,2.025000,1.333333,0,1.0,0
4,EVENT_0002,369260000,0.000000,4.842786,4.865497,-0.022711,0.000000,33.833333,27,9.240741,...,9.3,-0.1,1.781481,0.6,9.250000,9.235714,2.000000,0,1.0,1
5,EVENT_0002,538008071,4.252550,4.957861,4.252550,0.705311,29.150000,29.700000,26,1.696154,...,1.2,-0.9,4.403846,10.8,1.696154,1.200000,1.650000,0,1.0,0
6,EVENT_0003,249823000,3.086661,3.086661,4.920948,-1.834287,29.350000,4.800000,5,12.500000,...,12.5,0.0,0.400000,0.6,12.500000,12.500000,1.516667,0,1.0,0
7,EVENT_0003,353754000,0.375906,4.590701,1.349749,3.240952,25.450000,14.683333,13,12.923077,...,12.9,0.2,0.315385,0.0,12.890000,13.000000,1.350000,0,1.0,0
8,EVENT_0003,538003726,1.585724,4.870537,1.806982,3.063555,27.450000,14.000000,13,12.653846,...,12.7,0.0,0.323077,0.2,12.645455,12.700000,1.366667,0,1.0,0
9,EVENT_0003,538008391,0.000000,4.763452,4.906990,-0.143538,0.000000,23.516667,21,13.304762,...,13.2,0.4,0.690476,0.3,13.200000,13.400000,1.500000,0,1.0,1


In [38]:
# ============================================================
# PHASE 4.2 — CORRECTED TRAINING DATA GENERATION
# ============================================================

SOURCE_RADIUS_KM = 5
TIME_WINDOW_MINUTES = 30

training_candidate_rows = []


# ------------------------------------------------------------
# Process every controlled source event
# ------------------------------------------------------------

for _, event in source_event_df.iterrows():

    event_id = event["event_id"]

    source_mmsi = event["source_mmsi"]

    source_lat = event["source_lat"]
    source_lon = event["source_lon"]

    source_time = pd.to_datetime(
        event["source_time"],
        utc=True
    )

    # --------------------------------------------------------
    # 1. Find ALL AIS records in the ±30 minute window
    # --------------------------------------------------------

    time_difference = (
        ais["BaseDateTime"] - source_time
    ).abs()

    temporal_data = ais[
        time_difference <= pd.Timedelta(
            minutes=TIME_WINDOW_MINUTES
        )
    ].copy()

    # --------------------------------------------------------
    # 2. Calculate distance to estimated source
    # --------------------------------------------------------

    temporal_data["distance_to_source_km"] = (
        haversine_distance(
            source_lat,
            source_lon,
            temporal_data["LAT"].values,
            temporal_data["LON"].values
        )
    )

    # --------------------------------------------------------
    # 3. Find vessels that came within 5 km
    # --------------------------------------------------------

    candidate_mmsis = (
        temporal_data[
            temporal_data["distance_to_source_km"]
            <= SOURCE_RADIUS_KM
        ]["MMSI"]
        .dropna()
        .unique()
    )

    # --------------------------------------------------------
    # 4. Process each candidate vessel
    # --------------------------------------------------------

    for candidate_mmsi in candidate_mmsis:

        # IMPORTANT:
        # Use ALL of this candidate's points in the
        # ±30-minute temporal window, not just points
        # inside the 5-km radius.

        vessel_df = temporal_data[
            temporal_data["MMSI"] == candidate_mmsi
        ].copy()

        vessel_df = vessel_df.sort_values(
            "BaseDateTime"
        ).reset_index(drop=True)

        # Need at least two observations
        if len(vessel_df) < 2:
            continue

        # ----------------------------------------------------
        # Calculate final 18 features
        # ----------------------------------------------------

        features = calculate_candidate_features(
            vessel_df=vessel_df,
            source_lat=source_lat,
            source_lon=source_lon,
            source_time=source_time,
            source_radius_km=SOURCE_RADIUS_KM
        )

        # ----------------------------------------------------
        # Add event information
        # ----------------------------------------------------

        features["event_id"] = event_id
        features["MMSI"] = candidate_mmsi

        # ----------------------------------------------------
        # Create label
        # ----------------------------------------------------

        if candidate_mmsi == source_mmsi:
            features["label"] = 1
        else:
            features["label"] = 0

        training_candidate_rows.append(features)


# ============================================================
# Create DataFrame
# ============================================================

training_candidate_df = pd.DataFrame(
    training_candidate_rows
)


# ============================================================
# Final feature list
# ============================================================

feature_columns = [
    "closest_distance_km",
    "distance_at_window_start_km",
    "distance_at_window_end_km",
    "distance_change_km",

    "closest_time_difference_min",
    "time_inside_source_radius_min",
    "ais_point_count",

    "average_sog",
    "maximum_sog",
    "speed_at_closest_approach",
    "speed_change",

    "average_course_change",
    "course_change_at_closest_approach",
    "approach_speed",
    "departure_speed",

    "maximum_ais_gap_min",
    "long_ais_gap_count",
    "moving_fraction"
]


training_candidate_df = training_candidate_df[
    ["event_id", "MMSI"] +
    feature_columns +
    ["label"]
]


# ============================================================
# Summary
# ============================================================

print("CORRECTED TRAINING CANDIDATE DATASET")
print("=" * 70)

print(
    "Shape:",
    training_candidate_df.shape
)

print(
    "Unique events:",
    training_candidate_df["event_id"].nunique()
)

print(
    "Unique vessels:",
    training_candidate_df["MMSI"].nunique()
)

print("\nLabel distribution:")
print(
    training_candidate_df["label"].value_counts()
)

print("\nPositive ratio:")
print(
    training_candidate_df["label"].mean()
)

display(
    training_candidate_df.head(10)
)

CORRECTED TRAINING CANDIDATE DATASET
Shape: (312, 21)
Unique events: 100
Unique vessels: 155

Label distribution:
label
0    212
1    100
Name: count, dtype: int64

Positive ratio:
0.32051282051282054


,event_id,MMSI,closest_distance_km,distance_at_window_start_km,distance_at_window_end_km,distance_change_km,closest_time_difference_min,time_inside_source_radius_min,ais_point_count,average_sog,...,speed_at_closest_approach,speed_change,average_course_change,course_change_at_closest_approach,approach_speed,departure_speed,maximum_ais_gap_min,long_ais_gap_count,moving_fraction,label
0,EVENT_0001,367373000,3.512318,3.529057,3.522263,0.006793,26.883333,59.000000,49,0.002041,...,0.0,0.0,3.414286,0.0,0.002128,0.000000,1.516667,0,0.0,0
1,EVENT_0001,477288000,0.000000,12.777089,12.179865,0.597224,0.000000,22.700000,52,13.761538,...,13.8,0.8,1.192308,0.0,13.770370,13.753846,1.666667,0,1.0,1
2,EVENT_0001,538007823,2.594918,24.119527,2.594918,21.524609,29.383333,5.833333,49,12.108163,...,12.6,0.3,1.704082,0.4,12.108163,12.600000,1.833333,0,1.0,0
3,EVENT_0001,538008071,4.816536,4.846485,16.577662,-11.731177,28.800000,4.816667,49,7.097959,...,1.5,7.4,5.153061,47.7,1.500000,7.214583,1.833333,0,1.0,0
4,EVENT_0002,369260000,0.000000,8.355198,8.334929,0.020269,0.000000,33.833333,43,9.246512,...,9.3,0.3,1.958140,0.6,9.209524,9.282609,3.516667,0,1.0,1
5,EVENT_0002,538008071,4.252550,6.989966,4.252550,2.737416,29.150000,29.700000,50,2.118000,...,1.2,-3.3,4.926000,10.8,2.118000,1.200000,1.650000,0,1.0,0
6,EVENT_0003,249823000,3.086661,3.086661,25.909658,-22.822997,29.350000,4.800000,48,12.479167,...,12.5,0.1,0.543750,0.6,12.500000,12.479167,1.850000,0,1.0,0
7,EVENT_0003,353754000,0.375906,19.491661,1.349749,18.141912,25.450000,14.683333,51,12.290196,...,12.9,4.8,4.925490,0.0,12.243750,13.000000,1.483333,0,1.0,0
8,EVENT_0003,538003726,1.585724,20.576285,1.806982,18.769303,27.450000,14.000000,52,11.648077,...,12.7,2.9,0.430769,0.2,11.606000,12.700000,1.366667,0,1.0,0
9,EVENT_0003,538008391,0.000000,11.863762,11.911130,-0.047368,0.000000,23.516667,51,13.213725,...,13.2,0.6,1.080392,0.3,13.022222,13.420000,1.650000,0,1.0,1


In [39]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
from xgboost import XGBClassifier

# ---------------------------------------------------------
# 1. Features
# ---------------------------------------------------------

FEATURES = [
    "closest_distance_km",
    "distance_at_window_start_km",
    "distance_at_window_end_km",
    "distance_change_km",

    "closest_time_difference_min",
    "time_inside_source_radius_min",
    "ais_point_count",

    "average_sog",
    "maximum_sog",
    "speed_at_closest_approach",
    "speed_change",

    "average_course_change",
    "course_change_at_closest_approach",
    "approach_speed",
    "departure_speed",

    "maximum_ais_gap_min",
    "long_ais_gap_count",
    "moving_fraction"
]

# ---------------------------------------------------------
# 2. Prepare X and y
# ---------------------------------------------------------

X = training_candidate_df[FEATURES].copy()
y = training_candidate_df["label"].copy()

# Replace infinite values if any
X = X.replace([np.inf, -np.inf], np.nan)

# Check missing values
print("Missing values:")
print(X.isna().sum().sum())

# Fill any missing values
X = X.fillna(0)

# ---------------------------------------------------------
# 3. Split BY EVENT
# ---------------------------------------------------------

groups = training_candidate_df["event_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_events = groups.iloc[train_idx]
test_events = groups.iloc[test_idx]

print("\nTRAINING DATA")
print("=" * 60)
print("Rows:", len(X_train))
print("Events:", train_events.nunique())
print("Positive:", y_train.sum())
print("Negative:", (y_train == 0).sum())

print("\nTEST DATA")
print("=" * 60)
print("Rows:", len(X_test))
print("Events:", test_events.nunique())
print("Positive:", y_test.sum())
print("Negative:", (y_test == 0).sum())

print("\nEvent overlap:")
print(
    len(
        set(train_events.unique())
        & set(test_events.unique())
    )
)

Missing values:
0

TRAINING DATA
Rows: 235
Events: 80
Positive: 80
Negative: 155

TEST DATA
Rows: 77
Events: 20
Positive: 20
Negative: 57

Event overlap:
0


In [40]:
# ---------------------------------------------------------
# 4. Create XGBoost model
# ---------------------------------------------------------

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,

    objective="binary:logistic",
    eval_metric="logloss",

    random_state=42,
    n_jobs=-1
)

# ---------------------------------------------------------
# 5. Train
# ---------------------------------------------------------

model.fit(
    X_train,
    y_train
)

print("XGBoost training completed.")

XGBoost training completed.


In [41]:
# ---------------------------------------------------------
# 6. Predictions
# ---------------------------------------------------------

y_pred = model.predict(X_test)

y_probability = model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# 7. Evaluation
# ---------------------------------------------------------

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_probability)

print("\nXGBOOST RESULTS")
print("=" * 60)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

print("\nClassification Report")
print("=" * 60)

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

print("\nConfusion Matrix")
print("=" * 60)

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


XGBOOST RESULTS
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1 Score : 1.0000
ROC-AUC  : 1.0000

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        57
           1       1.00      1.00      1.00        20

    accuracy                           1.00        77
   macro avg       1.00      1.00      1.00        77
weighted avg       1.00      1.00      1.00        77


Confusion Matrix
[[57  0]
 [ 0 20]]


In [ ]:
# ---------------------------------------------------------
# FEATURE IMPORTANCE
# ---------------------------------------------------------

importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("XGBOOST FEATURE IMPORTANCE")
print("=" * 60)

display(importance_df)

In [42]:
# ---------------------------------------------------------
# TOP-1 / TOP-3 SOURCE VESSEL RANKING
# ---------------------------------------------------------

test_results = training_candidate_df.iloc[test_idx].copy()

test_results["predicted_probability"] = y_probability

# Rank candidates inside each event
test_results["rank"] = (
    test_results
    .groupby("event_id")["predicted_probability"]
    .rank(
        ascending=False,
        method="first"
    )
)

# ---------------------------------------------------------
# Calculate ranking performance
# ---------------------------------------------------------

top1_correct = (
    test_results[
        (test_results["label"] == 1) &
        (test_results["rank"] == 1)
    ]
    .shape[0]
)

top3_correct = (
    test_results[
        (test_results["label"] == 1) &
        (test_results["rank"] <= 3)
    ]
    .shape[0]
)

total_test_events = test_results["event_id"].nunique()

top1_accuracy = top1_correct / total_test_events
top3_accuracy = top3_correct / total_test_events

print("SOURCE VESSEL RANKING RESULTS")
print("=" * 60)

print("Test events:", total_test_events)
print("Top-1 correct:", top1_correct)
print("Top-3 correct:", top3_correct)

print(f"\nTop-1 Accuracy: {top1_accuracy:.4f}")
print(f"Top-3 Accuracy: {top3_accuracy:.4f}")

SOURCE VESSEL RANKING RESULTS
Test events: 20
Top-1 correct: 20
Top-3 correct: 20

Top-1 Accuracy: 1.0000
Top-3 Accuracy: 1.0000


In [43]:
# ---------------------------------------------------------
# PHASE 5.1 — INSPECT TEST EVENT RANKINGS
# ---------------------------------------------------------

ranking_view = test_results[
    [
        "event_id",
        "MMSI",
        "closest_distance_km",
        "closest_time_difference_min",
        "average_sog",
        "speed_change",
        "predicted_probability",
        "rank",
        "label"
    ]
].sort_values(
    ["event_id", "rank"]
).reset_index(drop=True)

print("TEST EVENT RANKINGS")
print("=" * 80)

display(ranking_view)

TEST EVENT RANKINGS


,event_id,MMSI,closest_distance_km,closest_time_difference_min,average_sog,speed_change,predicted_probability,rank,label
0,EVENT_0001,477288000,0.000000,0.000000,13.761538,0.8,0.984300,1.0,1
1,EVENT_0001,538008071,4.816536,28.800000,7.097959,7.4,0.007496,2.0,0
2,EVENT_0001,367373000,3.512318,26.883333,0.002041,0.0,0.007052,3.0,0
3,EVENT_0001,538007823,2.594918,29.383333,12.108163,0.3,0.007045,4.0,0
4,EVENT_0005,311050800,0.000000,0.000000,13.242857,3.9,0.984996,1.0,1
...,...,...,...,...,...,...,...,...,...
72,EVENT_0084,215031000,0.721334,27.166667,10.281481,0.2,0.007230,3.0,0
73,EVENT_0091,210347000,0.000000,0.000000,12.422449,1.2,0.987122,1.0,1
74,EVENT_0091,311035600,4.060878,6.233333,10.232653,-0.1,0.007919,2.0,0
75,EVENT_0091,354298000,1.889274,3.716667,15.849020,2.9,0.007766,3.0,0


In [44]:
# ---------------------------------------------------------
# PHASE 6 — REALISTIC CONTROLLED SOURCE EVENTS
# ---------------------------------------------------------

import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Settings
# ---------------------------------------------------------

NUMBER_OF_EVENTS = 100

# Maximum uncertainty introduced into the estimated origin
MAX_LOCATION_ERROR_KM = 2.0
MAX_TIME_ERROR_MINUTES = 15

RANDOM_STATE = 123

rng = np.random.default_rng(RANDOM_STATE)

# ---------------------------------------------------------
# Select possible real source points
# ---------------------------------------------------------

source_pool = ais[
    ais["SOG"] >= 2.0
].copy()

source_pool = source_pool.reset_index(drop=True)

# Select random real AIS points
source_indices = rng.choice(
    len(source_pool),
    size=min(NUMBER_OF_EVENTS, len(source_pool)),
    replace=False
)

actual_sources = source_pool.iloc[
    source_indices
].copy().reset_index(drop=True)

# ---------------------------------------------------------
# Create realistic estimated origins
# ---------------------------------------------------------

realistic_events = []

for i, row in actual_sources.iterrows():

    actual_lat = float(row["LAT"])
    actual_lon = float(row["LON"])
    actual_time = pd.to_datetime(
        row["BaseDateTime"],
        utc=True
    )

    # -----------------------------------------------------
    # Random location error
    # -----------------------------------------------------

    error_distance_km = rng.uniform(
        0.2,
        MAX_LOCATION_ERROR_KM
    )

    error_direction_deg = rng.uniform(
        0,
        360
    )

    direction_rad = np.radians(
        error_direction_deg
    )

    # Approximate conversion:
    # 1 degree latitude ≈ 111 km
    lat_error = (
        error_distance_km
        * np.cos(direction_rad)
        / 111.0
    )

    lon_error = (
        error_distance_km
        * np.sin(direction_rad)
        /
        (
            111.0
            * np.cos(np.radians(actual_lat))
        )
    )

    estimated_lat = actual_lat + lat_error
    estimated_lon = actual_lon + lon_error

    # -----------------------------------------------------
    # Random time error
    # -----------------------------------------------------

    time_error_minutes = rng.uniform(
        -MAX_TIME_ERROR_MINUTES,
        MAX_TIME_ERROR_MINUTES
    )

    estimated_time = (
        actual_time
        + pd.Timedelta(
            minutes=time_error_minutes
        )
    )

    # -----------------------------------------------------
    # Store event
    # -----------------------------------------------------

    realistic_events.append({
        "event_id": f"REALISTIC_EVENT_{i+1:04d}",

        # Actual source information
        "source_mmsi": row["MMSI"],
        "actual_source_lat": actual_lat,
        "actual_source_lon": actual_lon,
        "actual_source_time": actual_time,

        # Estimated origin used by the model
        "source_lat": estimated_lat,
        "source_lon": estimated_lon,
        "source_time": estimated_time,

        # Uncertainty information
        "location_error_km": error_distance_km,
        "time_error_minutes": abs(time_error_minutes)
    })

realistic_event_df = pd.DataFrame(
    realistic_events
)

# ---------------------------------------------------------
# Display results
# ---------------------------------------------------------

print("REALISTIC CONTROLLED SOURCE EVENTS")
print("=" * 70)

print(
    "Number of events:",
    len(realistic_event_df)
)

print(
    "Average location uncertainty:",
    round(
        realistic_event_df[
            "location_error_km"
        ].mean(),
        3
    ),
    "km"
)

print(
    "Maximum location uncertainty:",
    round(
        realistic_event_df[
            "location_error_km"
        ].max(),
        3
    ),
    "km"
)

print(
    "Average time uncertainty:",
    round(
        realistic_event_df[
            "time_error_minutes"
        ].mean(),
        3
    ),
    "minutes"
)

print(
    "Maximum time uncertainty:",
    round(
        realistic_event_df[
            "time_error_minutes"
        ].max(),
        3
    ),
    "minutes"
)

display(
    realistic_event_df.head(10)
)

REALISTIC CONTROLLED SOURCE EVENTS
Number of events: 100
Average location uncertainty: 1.048 km
Maximum location uncertainty: 1.989 km
Average time uncertainty: 7.588 minutes
Maximum time uncertainty: 14.76 minutes


,event_id,source_mmsi,actual_source_lat,actual_source_lon,actual_source_time,source_lat,source_lon,source_time,location_error_km,time_error_minutes
0,REALISTIC_EVENT_0001,303071000,28.55524,-89.42318,2021-12-31 06:24:46+00:00,28.545439,-89.417827,2021-12-31 06:35:31.475358806+00:00,1.206649,10.757923
1,REALISTIC_EVENT_0002,538007389,28.63817,-89.60713,2022-01-01 11:40:19+00:00,28.635351,-89.603266,2022-01-01 11:44:31.917886298+00:00,0.489496,4.215298
2,REALISTIC_EVENT_0003,538007981,28.68363,-89.19588,2022-01-01 00:48:27+00:00,28.682378,-89.176668,2022-01-01 00:54:05.577144447+00:00,1.875943,5.642952
3,REALISTIC_EVENT_0004,564966000,28.32547,-89.10567,2022-01-01 05:40:17+00:00,28.325209,-89.110454,2022-01-01 05:34:04.726309438+00:00,0.468295,6.204562
4,REALISTIC_EVENT_0005,367643990,28.84229,-89.63493,2022-01-01 05:51:09+00:00,28.843018,-89.628413,2022-01-01 05:47:16.172097302+00:00,0.638741,3.880465
5,REALISTIC_EVENT_0006,367124980,28.31943,-89.59310,2022-01-01 17:32:36+00:00,28.314013,-89.598365,2022-01-01 17:33:56.487720344+00:00,0.791399,1.341462
6,REALISTIC_EVENT_0007,367643110,28.85640,-89.54430,2021-12-31 02:54:01+00:00,28.852510,-89.528994,2021-12-31 03:07:49.996711075+00:00,1.549358,13.816612
7,REALISTIC_EVENT_0008,367590950,28.83275,-89.45522,2022-01-04 11:36:14+00:00,28.828196,-89.466165,2022-01-04 11:41:52.866647286+00:00,1.178269,5.647777
8,REALISTIC_EVENT_0009,215031000,28.66893,-89.33108,2021-12-31 23:14:52+00:00,28.671548,-89.327233,2021-12-31 23:13:43.678879690+00:00,0.474136,1.138685
9,REALISTIC_EVENT_0010,355899000,28.95832,-89.39192,2022-01-02 20:42:07+00:00,28.956903,-89.380300,2022-01-02 20:32:31.217704500+00:00,1.139480,9.596372


In [45]:
# ---------------------------------------------------------
# PHASE 6.1 — BUILD REALISTIC TRAINING CANDIDATES
# ---------------------------------------------------------

SOURCE_RADIUS_KM = 5
TIME_WINDOW_MINUTES = 30

realistic_training_rows = []

for _, event in realistic_event_df.iterrows():

    event_id = event["event_id"]
    source_mmsi = event["source_mmsi"]

    # IMPORTANT:
    # Use the UNCERTAIN estimated origin,
    # not the actual source position.
    source_lat = event["source_lat"]
    source_lon = event["source_lon"]
    source_time = pd.to_datetime(
        event["source_time"],
        utc=True
    )

    # -----------------------------------------------------
    # 1. Select AIS points within ±30 minutes
    # -----------------------------------------------------

    time_difference = (
        ais["BaseDateTime"] - source_time
    ).abs()

    temporal_data = ais[
        time_difference <= pd.Timedelta(
            minutes=TIME_WINDOW_MINUTES
        )
    ].copy()

    # -----------------------------------------------------
    # 2. Calculate distance from estimated origin
    # -----------------------------------------------------

    temporal_data["distance_to_source_km"] = (
        haversine_distance(
            source_lat,
            source_lon,
            temporal_data["LAT"].values,
            temporal_data["LON"].values
        )
    )

    # -----------------------------------------------------
    # 3. Find candidate vessels within 5 km
    # -----------------------------------------------------

    candidate_mmsis = (
        temporal_data[
            temporal_data["distance_to_source_km"]
            <= SOURCE_RADIUS_KM
        ]["MMSI"]
        .dropna()
        .unique()
    )

    # -----------------------------------------------------
    # 4. Calculate features for each candidate
    # -----------------------------------------------------

    for candidate_mmsi in candidate_mmsis:

        vessel_df = temporal_data[
            temporal_data["MMSI"] == candidate_mmsi
        ].copy()

        vessel_df = vessel_df.sort_values(
            "BaseDateTime"
        ).reset_index(drop=True)

        # Need enough points to calculate trajectory features
        if len(vessel_df) < 2:
            continue

        features = calculate_candidate_features(
            vessel_df=vessel_df,
            source_lat=source_lat,
            source_lon=source_lon,
            source_time=source_time,
            source_radius_km=SOURCE_RADIUS_KM
        )

        features["event_id"] = event_id
        features["MMSI"] = candidate_mmsi

        # -------------------------------------------------
        # Label
        # -------------------------------------------------
        #
        # The TRUE source vessel is known because this is
        # a controlled simulation.
        #
        # The model itself only sees the estimated origin.
        # -------------------------------------------------

        if candidate_mmsi == source_mmsi:
            features["label"] = 1
        else:
            features["label"] = 0

        realistic_training_rows.append(features)


# ---------------------------------------------------------
# 5. Create dataframe
# ---------------------------------------------------------

realistic_training_df = pd.DataFrame(
    realistic_training_rows
)

# ---------------------------------------------------------
# 6. Feature list
# ---------------------------------------------------------

feature_columns = [
    "closest_distance_km",
    "distance_at_window_start_km",
    "distance_at_window_end_km",
    "distance_change_km",

    "closest_time_difference_min",
    "time_inside_source_radius_min",
    "ais_point_count",

    "average_sog",
    "maximum_sog",
    "speed_at_closest_approach",
    "speed_change",

    "average_course_change",
    "course_change_at_closest_approach",
    "approach_speed",
    "departure_speed",

    "maximum_ais_gap_min",
    "long_ais_gap_count",
    "moving_fraction"
]

realistic_training_df = realistic_training_df[
    ["event_id", "MMSI"]
    + feature_columns
    + ["label"]
]

# ---------------------------------------------------------
# 7. Display dataset information
# ---------------------------------------------------------

print("REALISTIC TRAINING CANDIDATE DATASET")
print("=" * 70)

print(
    "Shape:",
    realistic_training_df.shape
)

print(
    "Unique events:",
    realistic_training_df["event_id"].nunique()
)

print(
    "Unique vessels:",
    realistic_training_df["MMSI"].nunique()
)

print("\nLabel distribution:")
print(
    realistic_training_df["label"].value_counts()
)

print("\nPositive ratio:")
print(
    realistic_training_df["label"].mean()
)

print("\nMissing values:")
print(
    realistic_training_df[
        feature_columns
    ].isna().sum().sum()
)

display(
    realistic_training_df.head(10)
)

REALISTIC TRAINING CANDIDATE DATASET
Shape: (314, 21)
Unique events: 100
Unique vessels: 142

Label distribution:
label
0    214
1    100
Name: count, dtype: int64

Positive ratio:
0.3184713375796178

Missing values:
0


,event_id,MMSI,closest_distance_km,distance_at_window_start_km,distance_at_window_end_km,distance_change_km,closest_time_difference_min,time_inside_source_radius_min,ais_point_count,average_sog,...,speed_at_closest_approach,speed_change,average_course_change,course_change_at_closest_approach,approach_speed,departure_speed,maximum_ais_gap_min,long_ais_gap_count,moving_fraction,label
0,REALISTIC_EVENT_0001,303071000,0.654576,5.095220,3.633670,1.461550,1.058744,55.166667,43,5.081395,...,2.6,-1.4,2.667442,2.3,5.727273,4.322727,2.316667,0,1.0,1
1,REALISTIC_EVENT_0002,219025316,3.798111,3.798111,21.565759,-17.767648,28.831965,3.483333,51,9.800000,...,10.1,0.8,0.409804,0.3,10.100000,9.800000,1.500000,0,1.0,0
2,REALISTIC_EVENT_0002,538007389,0.490360,7.822084,10.521660,-2.699576,4.215298,30.833333,51,10.086275,...,10.0,-0.1,0.766667,0.7,10.190909,10.006667,1.483333,0,1.0,1
3,REALISTIC_EVENT_0003,538007981,0.743143,5.281134,7.174941,-1.893806,11.642952,33.033333,48,6.785417,...,9.6,-7.7,1.262500,0.3,9.500000,5.670588,3.333333,0,1.0,1
4,REALISTIC_EVENT_0004,538005036,1.924590,13.684982,9.472588,4.212395,5.471228,21.133333,44,12.450000,...,12.5,-0.2,1.134091,2.4,12.481481,12.405556,3.666667,0,1.0,0
5,REALISTIC_EVENT_0004,564966000,0.469118,13.435411,9.058589,4.376821,6.204562,24.850000,48,12.229167,...,12.3,0.2,0.620833,0.0,12.258065,12.183333,4.500000,0,1.0,1
6,REALISTIC_EVENT_0005,367643990,0.389439,9.515059,8.221404,1.293655,1.730465,31.850000,46,9.763043,...,9.8,0.0,1.667391,0.5,9.788889,9.730000,5.016667,0,1.0,1
7,REALISTIC_EVENT_0006,311727000,2.926221,2.953696,3.648045,-0.694350,9.341462,49.166667,37,1.810811,...,1.3,0.4,5.183784,13.2,1.233333,1.972414,92.333333,1,1.0,0
8,REALISTIC_EVENT_0006,367124980,0.595887,1.092955,2.903972,-1.811017,2.658538,54.133333,22,4.059091,...,4.1,-0.6,11.813636,3.4,4.287500,3.940000,12.166667,0,1.0,1
9,REALISTIC_EVENT_0006,403522001,2.874287,2.906046,3.600126,-0.694080,8.758129,58.833333,51,1.509804,...,1.2,0.6,2.588235,1.0,0.983333,1.779412,16.516667,0,1.0,0


In [46]:
# ---------------------------------------------------------
# PHASE 6.2 — SANITY CHECK REALISTIC TRAINING DATA
# ---------------------------------------------------------

# Count positive source vessels in each event
positive_per_event = (
    realistic_training_df
    .groupby("event_id")["label"]
    .sum()
)

# Count total candidates in each event
candidates_per_event = (
    realistic_training_df
    .groupby("event_id")
    .size()
)

print("REALISTIC DATASET SANITY CHECK")
print("=" * 70)

print(
    "Total events:",
    realistic_training_df["event_id"].nunique()
)

print(
    "Events with exactly 1 positive:",
    (positive_per_event == 1).sum()
)

print(
    "Events with 0 positives:",
    (positive_per_event == 0).sum()
)

print(
    "Events with more than 1 positive:",
    (positive_per_event > 1).sum()
)

print("\nCandidate count per event:")
print(
    candidates_per_event.describe()
)

print("\nPositive count per event:")
print(
    positive_per_event.value_counts()
)

# ---------------------------------------------------------
# Check whether every true source MMSI is present
# ---------------------------------------------------------

event_source_check = (
    realistic_event_df[
        ["event_id", "source_mmsi"]
    ]
    .merge(
        realistic_training_df[
            ["event_id", "MMSI"]
        ],
        left_on=["event_id", "source_mmsi"],
        right_on=["event_id", "MMSI"],
        how="left",
        indicator=True
    )
)

missing_sources = event_source_check[
    event_source_check["_merge"] == "left_only"
]

print("\nTrue source vessels missing from candidates:")
print(
    len(missing_sources)
)

if len(missing_sources) > 0:
    display(missing_sources)
else:
    print("All true source vessels are present.")

REALISTIC DATASET SANITY CHECK
Total events: 100
Events with exactly 1 positive: 100
Events with 0 positives: 0
Events with more than 1 positive: 0

Candidate count per event:
count    100.000000
mean       3.140000
std        2.748811
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       12.000000
dtype: float64

Positive count per event:
label
1    100
Name: count, dtype: int64

True source vessels missing from candidates:
0
All true source vessels are present.


In [47]:
# ---------------------------------------------------------
# PHASE 6.3 — TRAIN XGBOOST ON REALISTIC EVENTS
# ---------------------------------------------------------

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
from xgboost import XGBClassifier

# ---------------------------------------------------------
# 1. Feature columns
# ---------------------------------------------------------

FEATURES = [
    "closest_distance_km",
    "distance_at_window_start_km",
    "distance_at_window_end_km",
    "distance_change_km",

    "closest_time_difference_min",
    "time_inside_source_radius_min",
    "ais_point_count",

    "average_sog",
    "maximum_sog",
    "speed_at_closest_approach",
    "speed_change",

    "average_course_change",
    "course_change_at_closest_approach",
    "approach_speed",
    "departure_speed",

    "maximum_ais_gap_min",
    "long_ais_gap_count",
    "moving_fraction"
]

# ---------------------------------------------------------
# 2. Prepare X and y
# ---------------------------------------------------------

X_realistic = realistic_training_df[
    FEATURES
].copy()

y_realistic = realistic_training_df[
    "label"
].copy()

# Replace infinite values
X_realistic = X_realistic.replace(
    [np.inf, -np.inf],
    np.nan
)

# Fill missing values if any
X_realistic = X_realistic.fillna(0)

print("Missing values:")
print(
    X_realistic.isna().sum().sum()
)

# ---------------------------------------------------------
# 3. Event-level train/test split
# ---------------------------------------------------------

groups_realistic = realistic_training_df[
    "event_id"
]

splitter_realistic = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx_realistic, test_idx_realistic = next(
    splitter_realistic.split(
        X_realistic,
        y_realistic,
        groups=groups_realistic
    )
)

X_train_realistic = X_realistic.iloc[
    train_idx_realistic
]

X_test_realistic = X_realistic.iloc[
    test_idx_realistic
]

y_train_realistic = y_realistic.iloc[
    train_idx_realistic
]

y_test_realistic = y_realistic.iloc[
    test_idx_realistic
]

train_events_realistic = groups_realistic.iloc[
    train_idx_realistic
]

test_events_realistic = groups_realistic.iloc[
    test_idx_realistic
]

# ---------------------------------------------------------
# 4. Print split information
# ---------------------------------------------------------

print("\nREALISTIC TRAINING DATA")
print("=" * 60)

print(
    "Rows:",
    len(X_train_realistic)
)

print(
    "Events:",
    train_events_realistic.nunique()
)

print(
    "Positive:",
    y_train_realistic.sum()
)

print(
    "Negative:",
    (y_train_realistic == 0).sum()
)

print("\nREALISTIC TEST DATA")
print("=" * 60)

print(
    "Rows:",
    len(X_test_realistic)
)

print(
    "Events:",
    test_events_realistic.nunique()
)

print(
    "Positive:",
    y_test_realistic.sum()
)

print(
    "Negative:",
    (y_test_realistic == 0).sum()
)

# ---------------------------------------------------------
# 5. Check event leakage
# ---------------------------------------------------------

event_overlap_realistic = (
    set(train_events_realistic.unique())
    &
    set(test_events_realistic.unique())
)

print("\nEvent overlap:")
print(
    len(event_overlap_realistic)
)

# ---------------------------------------------------------
# 6. Train XGBoost
# ---------------------------------------------------------

realistic_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,

    objective="binary:logistic",
    eval_metric="logloss",

    random_state=42,
    n_jobs=-1
)

realistic_model.fit(
    X_train_realistic,
    y_train_realistic
)

print("\nXGBoost training completed.")

Missing values:
0

REALISTIC TRAINING DATA
Rows: 253
Events: 80
Positive: 80
Negative: 173

REALISTIC TEST DATA
Rows: 61
Events: 20
Positive: 20
Negative: 41

Event overlap:
0

XGBoost training completed.


In [48]:
# ---------------------------------------------------------
# PHASE 6.4 — EVALUATE REALISTIC XGBOOST MODEL
# ---------------------------------------------------------

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Predictions
y_pred_realistic = realistic_model.predict(X_test_realistic)

# Probability that candidate is the source vessel
y_probability_realistic = realistic_model.predict_proba(
    X_test_realistic
)[:, 1]


# ---------------------------------------------------------
# METRICS
# ---------------------------------------------------------

accuracy_realistic = accuracy_score(
    y_test_realistic,
    y_pred_realistic
)

precision_realistic = precision_score(
    y_test_realistic,
    y_pred_realistic
)

recall_realistic = recall_score(
    y_test_realistic,
    y_pred_realistic
)

f1_realistic = f1_score(
    y_test_realistic,
    y_pred_realistic
)

roc_auc_realistic = roc_auc_score(
    y_test_realistic,
    y_probability_realistic
)


print("REALISTIC XGBOOST TEST RESULTS")
print("=" * 60)

print(f"Accuracy : {accuracy_realistic:.4f}")
print(f"Precision: {precision_realistic:.4f}")
print(f"Recall   : {recall_realistic:.4f}")
print(f"F1 Score : {f1_realistic:.4f}")
print(f"ROC-AUC  : {roc_auc_realistic:.4f}")


# ---------------------------------------------------------
# CLASSIFICATION REPORT
# ---------------------------------------------------------

print("\nCLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test_realistic,
        y_pred_realistic,
        target_names=[
            "Non-source vessel",
            "Potential source vessel"
        ]
    )
)


# ---------------------------------------------------------
# CONFUSION MATRIX
# ---------------------------------------------------------

print("\nCONFUSION MATRIX")
print("=" * 60)

cm_realistic = confusion_matrix(
    y_test_realistic,
    y_pred_realistic
)

print(cm_realistic)

REALISTIC XGBOOST TEST RESULTS
Accuracy : 0.7869
Precision: 0.7059
Recall   : 0.6000
F1 Score : 0.6486
ROC-AUC  : 0.9098

CLASSIFICATION REPORT
                         precision    recall  f1-score   support

      Non-source vessel       0.82      0.88      0.85        41
Potential source vessel       0.71      0.60      0.65        20

               accuracy                           0.79        61
              macro avg       0.76      0.74      0.75        61
           weighted avg       0.78      0.79      0.78        61


CONFUSION MATRIX
[[36  5]
 [ 8 12]]


In [49]:
# ---------------------------------------------------------
# PHASE 6.5 — FEATURE IMPORTANCE + VESSEL RANKING
# ---------------------------------------------------------

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# PART A — FEATURE IMPORTANCE
# ---------------------------------------------------------

importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance": realistic_model.feature_importances_
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("XGBOOST FEATURE IMPORTANCE")
print("=" * 60)

print(importance_df.to_string(index=False))


# ---------------------------------------------------------
# PART B — PREPARE TEST DATA WITH PREDICTIONS
# ---------------------------------------------------------

test_results = realistic_training_df.iloc[test_idx_realistic].copy()

test_results["source_probability"] = y_probability_realistic

test_results["predicted_class"] = y_pred_realistic


# ---------------------------------------------------------
# PART C — RANK CANDIDATES WITHIN EACH EVENT
# ---------------------------------------------------------

test_results["rank"] = (
    test_results
    .groupby("event_id")["source_probability"]
    .rank(
        ascending=False,
        method="first"
    )
)

# Number of candidates in each event
test_results["candidate_count"] = (
    test_results
    .groupby("event_id")["event_id"]
    .transform("count")
)


# ---------------------------------------------------------
# PART D — TOP-1 / TOP-3 PERFORMANCE
# ---------------------------------------------------------

event_summary = []

for event_id, group in test_results.groupby("event_id"):

    group = group.sort_values(
        "source_probability",
        ascending=False
    )

    true_source = group["label"] == 1

    source_rank = group.loc[
        true_source,
        "rank"
    ].iloc[0]

    candidate_count = len(group)

    event_summary.append({
        "event_id": event_id,
        "candidate_count": candidate_count,
        "source_rank": int(source_rank)
    })


ranking_df = pd.DataFrame(event_summary)


# ---------------------------------------------------------
# ALL EVENTS
# ---------------------------------------------------------

top1_all = (
    ranking_df["source_rank"] == 1
).mean()

top3_all = (
    ranking_df["source_rank"] <= 3
).mean()


print("\nRANKING PERFORMANCE — ALL TEST EVENTS")
print("=" * 60)

print(f"Test events : {len(ranking_df)}")
print(f"Top-1       : {top1_all:.2%}")
print(f"Top-3       : {top3_all:.2%}")


# ---------------------------------------------------------
# NON-TRIVIAL EVENTS
# ONLY EVENTS WITH 2 OR MORE CANDIDATES
# ---------------------------------------------------------

non_trivial = ranking_df[
    ranking_df["candidate_count"] >= 2
].copy()

top1_nontrivial = (
    non_trivial["source_rank"] == 1
).mean()

top3_nontrivial = (
    non_trivial["source_rank"] <= 3
).mean()


print("\nRANKING PERFORMANCE — EVENTS WITH >= 2 CANDIDATES")
print("=" * 60)

print(f"Events      : {len(non_trivial)}")
print(f"Top-1       : {top1_nontrivial:.2%}")
print(f"Top-3       : {top3_nontrivial:.2%}")


# ---------------------------------------------------------
# PART E — SHOW EVENT-BY-EVENT RANKING
# ---------------------------------------------------------

print("\nEVENT-BY-EVENT SOURCE RANKING")
print("=" * 60)

print(
    ranking_df.sort_values("event_id").to_string(index=False)
)

XGBOOST FEATURE IMPORTANCE
                          feature  importance
              closest_distance_km    0.171693
      closest_time_difference_min    0.132754
                  moving_fraction    0.104744
              maximum_ais_gap_min    0.061807
        speed_at_closest_approach    0.053374
                      average_sog    0.050557
                  departure_speed    0.047097
      distance_at_window_start_km    0.043605
                  ais_point_count    0.041086
                      maximum_sog    0.040860
            average_course_change    0.038690
    time_inside_source_radius_min    0.038662
               distance_change_km    0.037650
                   approach_speed    0.035915
                     speed_change    0.035088
course_change_at_closest_approach    0.034175
        distance_at_window_end_km    0.032243
               long_ais_gap_count    0.000000

RANKING PERFORMANCE — ALL TEST EVENTS
Test events : 20
Top-1       : 80.00%
Top-3       : 100.00%


In [50]:
# ---------------------------------------------------------
# PHASE 6.6 — INSPECT MIS-RANKED SOURCE EVENTS
# CORRECTED VERSION
# ---------------------------------------------------------

misranked_events = ranking_df[
    ranking_df["source_rank"] > 1
]["event_id"].tolist()

print("MIS-RANKED EVENTS")
print("=" * 60)
print(misranked_events)


# Show available columns so we know exactly what is present
print("\nAvailable columns in test_results:")
print(test_results.columns.tolist())


for event_id in misranked_events:

    print("\n")
    print("=" * 80)
    print(event_id)
    print("=" * 80)

    event_data = test_results[
        test_results["event_id"] == event_id
    ].copy()

    event_data = event_data.sort_values(
        "source_probability",
        ascending=False
    )

    # Only use columns that actually exist
    columns_to_show = [
        "event_id",
        "label",
        "source_probability",
        "rank",
        "candidate_count",

        "closest_distance_km",
        "closest_time_difference_min",

        "moving_fraction",
        "speed_at_closest_approach",
        "average_sog",
        "maximum_sog",

        "departure_speed",
        "approach_speed",
        "speed_change",

        "average_course_change",
        "course_change_at_closest_approach",

        "ais_point_count",
        "time_inside_source_radius_min",

        "distance_at_window_start_km",
        "distance_at_window_end_km",
        "distance_change_km"
    ]

    # Keep only columns that exist
    columns_to_show = [
        col for col in columns_to_show
        if col in event_data.columns
    ]

    print(
        event_data[columns_to_show].to_string(
            index=False
        )
    )

MIS-RANKED EVENTS
['REALISTIC_EVENT_0011', 'REALISTIC_EVENT_0040', 'REALISTIC_EVENT_0074', 'REALISTIC_EVENT_0081']

Available columns in test_results:
['event_id', 'MMSI', 'closest_distance_km', 'distance_at_window_start_km', 'distance_at_window_end_km', 'distance_change_km', 'closest_time_difference_min', 'time_inside_source_radius_min', 'ais_point_count', 'average_sog', 'maximum_sog', 'speed_at_closest_approach', 'speed_change', 'average_course_change', 'course_change_at_closest_approach', 'approach_speed', 'departure_speed', 'maximum_ais_gap_min', 'long_ais_gap_count', 'moving_fraction', 'label', 'source_probability', 'predicted_class', 'rank', 'candidate_count']


REALISTIC_EVENT_0011
            event_id  label  source_probability  rank  candidate_count  closest_distance_km  closest_time_difference_min  moving_fraction  speed_at_closest_approach  average_sog  maximum_sog  departure_speed  approach_speed  speed_change  average_course_change  course_change_at_closest_approach  ais_p

In [51]:
# ---------------------------------------------------------
# PHASE 7.1 — GENERATE LARGER REALISTIC EVENT DATASET
# VERSION 3
# ---------------------------------------------------------

import numpy as np
import pandas as pd

np.random.seed(42)

NUM_EVENTS_V3 = 300

realistic_events_v3 = []

# Use moving AIS points as possible real source events
moving_points_v3 = ais[
    ais["SOG"] >= 2
].copy()

moving_points_v3 = moving_points_v3.sample(
    n=NUM_EVENTS_V3,
    random_state=42
).reset_index(drop=True)


for i, row in moving_points_v3.iterrows():

    actual_lat = row["LAT"]
    actual_lon = row["LON"]
    actual_time = pd.to_datetime(
        row["BaseDateTime"],
        utc=True
    )

    # -----------------------------------------------------
    # Add uncertainty to estimated source
    # -----------------------------------------------------

    location_error_km = np.random.uniform(
        0.2,
        2.0
    )

    time_error_min = np.random.uniform(
        -15,
        15
    )

    # Random direction for location uncertainty
    angle = np.random.uniform(
        0,
        2 * np.pi
    )

    # Convert km to approximate latitude/longitude shift
    lat_shift = (
        location_error_km * np.cos(angle) / 111.0
    )

    lon_shift = (
        location_error_km
        * np.sin(angle)
        / (
            111.0
            * np.cos(np.radians(actual_lat))
        )
    )

    estimated_lat = actual_lat + lat_shift
    estimated_lon = actual_lon + lon_shift

    estimated_time = (
        actual_time
        + pd.Timedelta(
            minutes=time_error_min
        )
    )

    realistic_events_v3.append({

        "event_id":
            f"V3_EVENT_{i+1:04d}",

        # Actual source — used only for labels/evaluation
        "actual_source_mmsi":
            row["MMSI"],

        "actual_source_lat":
            actual_lat,

        "actual_source_lon":
            actual_lon,

        "actual_source_time":
            actual_time,

        # Estimated source — what the AIS model receives
        "source_lat":
            estimated_lat,

        "source_lon":
            estimated_lon,

        "source_time":
            estimated_time,

        "location_error_km":
            location_error_km,

        "time_error_min":
            abs(time_error_min)
    })


realistic_event_df_v3 = pd.DataFrame(
    realistic_events_v3
)


print("VERSION 3 REALISTIC EVENTS")
print("=" * 60)

print(
    "Number of events:",
    len(realistic_event_df_v3)
)

print(
    "Average location uncertainty:",
    round(
        realistic_event_df_v3[
            "location_error_km"
        ].mean(),
        3
    ),
    "km"
)

print(
    "Maximum location uncertainty:",
    round(
        realistic_event_df_v3[
            "location_error_km"
        ].max(),
        3
    ),
    "km"
)

print(
    "Average time uncertainty:",
    round(
        realistic_event_df_v3[
            "time_error_min"
        ].mean(),
        3
    ),
    "minutes"
)

print(
    "Maximum time uncertainty:",
    round(
        realistic_event_df_v3[
            "time_error_min"
        ].max(),
        3
    ),
    "minutes"
)

print("\nFirst 5 events:")
display(
    realistic_event_df_v3.head()
)

VERSION 3 REALISTIC EVENTS
Number of events: 300
Average location uncertainty: 1.111 km
Maximum location uncertainty: 1.999 km
Average time uncertainty: 7.843 minutes
Maximum time uncertainty: 14.906 minutes

First 5 events:


,event_id,actual_source_mmsi,actual_source_lat,actual_source_lon,actual_source_time,source_lat,source_lon,source_time,location_error_km,time_error_min
0,V3_EVENT_0001,477288000,28.69323,-89.55078,2022-01-03 08:34:51+00:00,28.692341,-89.559700,2022-01-03 08:48:22.285751537+00:00,0.874172,13.521429
1,V3_EVENT_0002,369260000,28.77401,-89.66129,2022-01-02 13:56:46+00:00,28.780420,-89.650384,2022-01-02 13:46:26.833552797+00:00,1.277585,10.319441
2,V3_EVENT_0003,538008391,28.74151,-89.51769,2022-01-01 00:28:10+00:00,28.739302,-89.519547,2022-01-01 00:39:09.117062394+00:00,0.304551,10.985284
3,V3_EVENT_0004,538009555,28.55262,-89.21176,2022-01-04 19:41:13+00:00,28.565667,-89.214602,2022-01-04 19:26:50.052089733+00:00,1.474531,14.382465
4,V3_EVENT_0005,311050800,28.94085,-89.40408,2022-01-02 03:28:57+00:00,28.947206,-89.388175,2022-01-02 03:20:19.210399221+00:00,1.698397,8.629827


In [52]:
# ---------------------------------------------------------
# PHASE 7.2 — BUILD VERSION 3 CANDIDATE DATASET
# ---------------------------------------------------------

SOURCE_RADIUS_KM = 5
TIME_WINDOW_MIN = 30

v3_training_rows = []

for _, event in realistic_event_df_v3.iterrows():

    event_id = event["event_id"]

    source_lat = event["source_lat"]
    source_lon = event["source_lon"]
    source_time = pd.to_datetime(
        event["source_time"],
        utc=True
    )

    actual_source_mmsi = event["actual_source_mmsi"]

    # -----------------------------------------------------
    # Temporal window around estimated source time
    # -----------------------------------------------------

    time_start = (
        source_time
        - pd.Timedelta(minutes=TIME_WINDOW_MIN)
    )

    time_end = (
        source_time
        + pd.Timedelta(minutes=TIME_WINDOW_MIN)
    )

    event_window = ais[
        (ais["BaseDateTime"] >= time_start) &
        (ais["BaseDateTime"] <= time_end)
    ].copy()

    if event_window.empty:
        continue

    # -----------------------------------------------------
    # Calculate distance from estimated source
    # -----------------------------------------------------

    event_window["distance_to_source_km"] = (
        haversine_distance(
            event_window["LAT"].values,
            event_window["LON"].values,
            source_lat,
            source_lon
        )
    )

    # -----------------------------------------------------
    # Find vessels that came within source radius
    # -----------------------------------------------------

    candidate_mmsis = (
        event_window[
            event_window["distance_to_source_km"]
            <= SOURCE_RADIUS_KM
        ]["MMSI"]
        .unique()
    )

    # -----------------------------------------------------
    # Extract complete temporal window
    # for every candidate vessel
    # -----------------------------------------------------

    for mmsi in candidate_mmsis:

        vessel_window = event_window[
            event_window["MMSI"] == mmsi
        ].copy()

        if vessel_window.empty:
            continue

        # -------------------------------------------------
        # Calculate the 18 model features
        # -------------------------------------------------

        features = calculate_candidate_features(
            vessel_window,
            source_lat,
            source_lon,
            source_time,
            source_radius_km=SOURCE_RADIUS_KM
        )

        # -------------------------------------------------
        # Create training label
        # -------------------------------------------------

        label = int(
            mmsi == actual_source_mmsi
        )

        row = {
            "event_id": event_id,
            "MMSI": mmsi,
            **features,
            "label": label
        }

        v3_training_rows.append(row)


# ---------------------------------------------------------
# CREATE DATAFRAME
# ---------------------------------------------------------

realistic_training_df_v3 = pd.DataFrame(
    v3_training_rows
)


# ---------------------------------------------------------
# CLEAN NUMERIC VALUES
# ---------------------------------------------------------

realistic_training_df_v3 = (
    realistic_training_df_v3
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)


# ---------------------------------------------------------
# RESULTS
# ---------------------------------------------------------

print("VERSION 3 REALISTIC TRAINING DATA")
print("=" * 60)

print(
    "Shape:",
    realistic_training_df_v3.shape
)

print(
    "Unique events:",
    realistic_training_df_v3[
        "event_id"
    ].nunique()
)

print(
    "Unique vessels:",
    realistic_training_df_v3[
        "MMSI"
    ].nunique()
)

print("\nLabel distribution:")
print(
    realistic_training_df_v3[
        "label"
    ].value_counts()
)

print("\nPositive ratio:")
print(
    realistic_training_df_v3[
        "label"
    ].mean()
)

print("\nMissing values:")
print(
    realistic_training_df_v3.isna().sum().sum()
)

VERSION 3 REALISTIC TRAINING DATA
Shape: (970, 21)
Unique events: 300
Unique vessels: 251

Label distribution:
label
0    670
1    300
Name: count, dtype: int64

Positive ratio:
0.30927835051546393

Missing values:
0


In [53]:
# ---------------------------------------------------------
# PHASE 7.3 — VERSION 3 DATASET SANITY CHECK
# ---------------------------------------------------------

# Number of candidates for each event
candidate_counts_v3 = (
    realistic_training_df_v3
    .groupby("event_id")
    .size()
)

# Number of positive/source vessels for each event
positive_counts_v3 = (
    realistic_training_df_v3
    .groupby("event_id")["label"]
    .sum()
)

print("VERSION 3 DATASET SANITY CHECK")
print("=" * 60)

print("\nCandidate count per event:")
print(candidate_counts_v3.describe())

print("\nEvents with exactly 1 candidate:")
print((candidate_counts_v3 == 1).sum())

print("\nEvents with 2+ candidates:")
print((candidate_counts_v3 >= 2).sum())

print("\nEvents with 3+ candidates:")
print((candidate_counts_v3 >= 3).sum())

print("\nEvents with 5+ candidates:")
print((candidate_counts_v3 >= 5).sum())

print("\nPositive source count per event:")
print(positive_counts_v3.value_counts().sort_index())

print("\nEvents with exactly one positive source:")
print((positive_counts_v3 == 1).sum())

print("\nEvents with zero positive sources:")
print((positive_counts_v3 == 0).sum())

print("\nEvents with multiple positive sources:")
print((positive_counts_v3 > 1).sum())

VERSION 3 DATASET SANITY CHECK

Candidate count per event:
count    300.000000
mean       3.233333
std        2.554861
min        1.000000
25%        1.000000
50%        2.000000
75%        4.250000
max       12.000000
dtype: float64

Events with exactly 1 candidate:
97

Events with 2+ candidates:
203

Events with 3+ candidates:
138

Events with 5+ candidates:
75

Positive source count per event:
label
1    300
Name: count, dtype: int64

Events with exactly one positive source:
300

Events with zero positive sources:
0

Events with multiple positive sources:
0


In [54]:
# ---------------------------------------------------------
# PHASE 7.4 — VERSION 3 XGBOOST TRAINING
# ---------------------------------------------------------

from xgboost import XGBClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

FEATURES = [
    "closest_distance_km",
    "distance_at_window_start_km",
    "distance_at_window_end_km",
    "distance_change_km",

    "closest_time_difference_min",
    "time_inside_source_radius_min",
    "ais_point_count",

    "average_sog",
    "maximum_sog",
    "speed_at_closest_approach",
    "speed_change",

    "average_course_change",
    "course_change_at_closest_approach",
    "approach_speed",
    "departure_speed",

    "maximum_ais_gap_min",
    "long_ais_gap_count",
    "moving_fraction"
]

X = realistic_training_df_v3[FEATURES]
y = realistic_training_df_v3["label"]
groups = realistic_training_df_v3["event_id"]

# ---------------------------------------------------------
# EVENT-LEVEL TRAIN/TEST SPLIT
# ---------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_events = groups.iloc[train_idx]
test_events = groups.iloc[test_idx]

print("VERSION 3 TRAIN / TEST SPLIT")
print("=" * 60)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("Training events:", train_events.nunique())
print("Testing events:", test_events.nunique())

print(
    "Event overlap:",
    len(set(train_events) & set(test_events))
)

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

# ---------------------------------------------------------
# XGBOOST MODEL
# ---------------------------------------------------------

realistic_model_v3 = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

realistic_model_v3.fit(
    X_train,
    y_train
)

print("\nXGBoost training completed.")

VERSION 3 TRAIN / TEST SPLIT
Training rows: 782
Testing rows: 188
Training events: 240
Testing events: 60
Event overlap: 0

Training label distribution:
label
0    542
1    240
Name: count, dtype: int64

Testing label distribution:
label
0    128
1     60
Name: count, dtype: int64

XGBoost training completed.


In [55]:
# ---------------------------------------------------------
# PHASE 7.5 — VERSION 3 XGBOOST EVALUATION
# ---------------------------------------------------------

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# ---------------------------------------------------------
# 1. CLASSIFICATION PREDICTIONS
# ---------------------------------------------------------

y_pred = realistic_model_v3.predict(X_test)

y_prob = realistic_model_v3.predict_proba(X_test)[:, 1]

print("VERSION 3 CLASSIFICATION RESULTS")
print("=" * 60)

print(
    "Accuracy:",
    round(accuracy_score(y_test, y_pred), 4)
)

print(
    "Precision:",
    round(precision_score(y_test, y_pred), 4)
)

print(
    "Recall:",
    round(recall_score(y_test, y_pred), 4)
)

print(
    "F1 Score:",
    round(f1_score(y_test, y_pred), 4)
)

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, y_prob), 4)
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Non-source",
            "Potential source"
        ]
    )
)

# ---------------------------------------------------------
# 2. BUILD TEST RESULTS TABLE
# ---------------------------------------------------------

test_results_v3 = realistic_training_df_v3.iloc[
    test_idx
].copy()

test_results_v3["predicted_probability"] = y_prob
test_results_v3["predicted_label"] = y_pred

print("\nTest result rows:", len(test_results_v3))

VERSION 3 CLASSIFICATION RESULTS
Accuracy: 0.867
Precision: 0.8182
Recall: 0.75
F1 Score: 0.7826
ROC-AUC: 0.9473

Confusion Matrix:
[[118  10]
 [ 15  45]]

Classification Report:
                  precision    recall  f1-score   support

      Non-source       0.89      0.92      0.90       128
Potential source       0.82      0.75      0.78        60

        accuracy                           0.87       188
       macro avg       0.85      0.84      0.84       188
    weighted avg       0.87      0.87      0.87       188


Test result rows: 188


In [56]:
# ---------------------------------------------------------
# PHASE 7.6 — VERSION 3 VESSEL RANKING
# ---------------------------------------------------------

# Rank candidates inside each event
ranking_results_v3 = []

for event_id, group in test_results_v3.groupby("event_id"):

    group = group.copy()

    # Highest XGBoost probability first
    group = group.sort_values(
        "predicted_probability",
        ascending=False
    ).reset_index(drop=True)

    # Rank: 1 = highest probability
    group["model_rank"] = np.arange(1, len(group) + 1)

    ranking_results_v3.append(group)


ranking_results_v3 = pd.concat(
    ranking_results_v3,
    ignore_index=True
)

# ---------------------------------------------------------
# FIND RANK OF TRUE SOURCE
# ---------------------------------------------------------

source_ranks_v3 = (
    ranking_results_v3[
        ranking_results_v3["label"] == 1
    ]
    .groupby("event_id")["model_rank"]
    .first()
)

# ---------------------------------------------------------
# TOP-K PERFORMANCE
# ---------------------------------------------------------

total_events = len(source_ranks_v3)

top1 = (source_ranks_v3 <= 1).sum()
top3 = (source_ranks_v3 <= 3).sum()
top5 = (source_ranks_v3 <= 5).sum()

print("VERSION 3 VESSEL RANKING RESULTS")
print("=" * 60)

print("Total test events:", total_events)

print(
    "Top-1:",
    top1,
    "/",
    total_events,
    "=",
    round(top1 / total_events * 100, 2),
    "%"
)

print(
    "Top-3:",
    top3,
    "/",
    total_events,
    "=",
    round(top3 / total_events * 100, 2),
    "%"
)

print(
    "Top-5:",
    top5,
    "/",
    total_events,
    "=",
    round(top5 / total_events * 100, 2),
    "%"
)

# ---------------------------------------------------------
# MULTI-CANDIDATE EVENTS ONLY
# ---------------------------------------------------------

candidate_counts_test_v3 = (
    ranking_results_v3
    .groupby("event_id")
    .size()
)

multi_candidate_events = (
    candidate_counts_test_v3 >= 2
)

multi_event_ids = (
    candidate_counts_test_v3[
        multi_candidate_events
    ].index
)

multi_source_ranks = source_ranks_v3[
    source_ranks_v3.index.isin(multi_event_ids)
]

multi_total = len(multi_source_ranks)

multi_top1 = (
    multi_source_ranks <= 1
).sum()

multi_top3 = (
    multi_source_ranks <= 3
).sum()

multi_top5 = (
    multi_source_ranks <= 5
).sum()

print("\nMULTI-CANDIDATE EVENTS")
print("=" * 60)

print("Multi-candidate events:", multi_total)

print(
    "Top-1:",
    multi_top1,
    "/",
    multi_total,
    "=",
    round(multi_top1 / multi_total * 100, 2),
    "%"
)

print(
    "Top-3:",
    multi_top3,
    "/",
    multi_total,
    "=",
    round(multi_top3 / multi_total * 100, 2),
    "%"
)

print(
    "Top-5:",
    multi_top5,
    "/",
    multi_total,
    "=",
    round(multi_top5 / multi_total * 100, 2),
    "%"
)

# ---------------------------------------------------------
# RANK DISTRIBUTION
# ---------------------------------------------------------

print("\nSOURCE RANK DISTRIBUTION")
print("=" * 60)

print(
    source_ranks_v3
    .value_counts()
    .sort_index()
)


VERSION 3 VESSEL RANKING RESULTS
Total test events: 60
Top-1: 55 / 60 = 91.67 %
Top-3: 60 / 60 = 100.0 %
Top-5: 60 / 60 = 100.0 %

MULTI-CANDIDATE EVENTS
Multi-candidate events: 41
Top-1: 36 / 41 = 87.8 %
Top-3: 41 / 41 = 100.0 %
Top-5: 41 / 41 = 100.0 %

SOURCE RANK DISTRIBUTION
model_rank
1    55
2     4
3     1
Name: count, dtype: int64


In [57]:
# ---------------------------------------------------------
# PHASE 7.7 — INSPECT VERSION 3 MISRANKED EVENTS
# ---------------------------------------------------------

misranked_events_v3 = source_ranks_v3[
    source_ranks_v3 > 1
]

print("MIS-RANKED TEST EVENTS")
print("=" * 60)

print(
    "Number of mis-ranked events:",
    len(misranked_events_v3)
)

print("\nEvent → True source rank:")
print(misranked_events_v3)


MIS-RANKED TEST EVENTS
Number of mis-ranked events: 5

Event → True source rank:
event_id
V3_EVENT_0114    2
V3_EVENT_0166    3
V3_EVENT_0176    2
V3_EVENT_0212    2
V3_EVENT_0234    2
Name: model_rank, dtype: int64


In [58]:
# ---------------------------------------------------------
# PHASE 7.8 — INSPECT THE 5 MIS-RANKED EVENTS
# ---------------------------------------------------------

misranked_ids_v3 = misranked_events_v3.index.tolist()

for event_id in misranked_ids_v3:

    print("\n" + "=" * 90)
    print("EVENT:", event_id)
    print("=" * 90)

    event_candidates = (
        ranking_results_v3[
            ranking_results_v3["event_id"] == event_id
        ]
        .sort_values("model_rank")
    )

    display(
        event_candidates[
            [
                "MMSI",
                "label",
                "model_rank",
                "predicted_probability",
                "closest_distance_km",
                "closest_time_difference_min",
                "distance_at_window_start_km",
                "distance_at_window_end_km",
                "distance_change_km",
                "time_inside_source_radius_min",
                "ais_point_count",
                "average_sog",
                "maximum_sog",
                "speed_at_closest_approach",
                "speed_change",
                "average_course_change",
                "course_change_at_closest_approach",
                "approach_speed",
                "departure_speed",
                "maximum_ais_gap_min",
                "long_ais_gap_count",
                "moving_fraction"
            ]
        ]
    )


EVENT: V3_EVENT_0114


,MMSI,label,model_rank,predicted_probability,closest_distance_km,closest_time_difference_min,distance_at_window_start_km,distance_at_window_end_km,distance_change_km,time_inside_source_radius_min,...,maximum_sog,speed_at_closest_approach,speed_change,average_course_change,course_change_at_closest_approach,approach_speed,departure_speed,maximum_ais_gap_min,long_ais_gap_count,moving_fraction
69,367007750,0,1,0.958855,0.667826,10.348476,3.869586,2.531197,1.338389,55.650000,...,9.0,2.4,-8.0,32.719149,47.3,7.686667,1.724242,474.333333,1,0.702128
70,367642410,1,2,0.880044,0.218002,2.584857,1.825751,1.260012,0.565739,57.500000,...,3.5,3.5,-0.4,44.222449,2.3,2.114815,1.904348,1.683333,0,0.775510
71,367456430,0,3,0.485600,0.213715,21.865143,0.649664,1.970201,-1.320537,58.150000,...,5.7,3.8,3.8,38.078000,8.7,2.428571,2.056818,1.516667,0,0.640000
72,367330830,0,4,0.018403,4.848013,11.868190,6.514744,5.241254,1.273490,23.333333,...,3.7,3.3,-0.1,0.925490,0.8,3.488889,3.518750,1.350000,0,1.000000
73,367676470,0,5,0.004684,2.530829,0.568190,3.200911,3.226910,-0.025999,58.166667,...,4.5,1.7,-0.5,50.432653,174.3,0.992000,1.016000,1.516667,0,0.346939



EVENT: V3_EVENT_0166


,MMSI,label,model_rank,predicted_probability,closest_distance_km,closest_time_difference_min,distance_at_window_start_km,distance_at_window_end_km,distance_change_km,time_inside_source_radius_min,...,maximum_sog,speed_at_closest_approach,speed_change,average_course_change,course_change_at_closest_approach,approach_speed,departure_speed,maximum_ais_gap_min,long_ais_gap_count,moving_fraction
89,538008519,0,1,0.899123,0.055795,1.040317,8.463412,4.297643,4.165770,30.016667,...,10.2,10.0,0.0,3.694595,2.1,9.484615,10.083333,50.483333,1,1.000000
90,368502516,0,2,0.787801,0.113003,4.076350,4.109185,13.276489,-9.167304,19.483333,...,18.2,16.6,0.0,8.278788,0.1,16.637500,15.253846,2.000000,0,1.000000
91,368207090,1,3,0.190269,0.147017,1.476350,5.950728,0.805389,5.145339,25.483333,...,29.2,23.2,-24.7,49.208889,3.2,23.648276,5.452941,1.650000,0,0.844444
92,636017782,0,4,0.016423,0.151488,26.543017,0.696425,4.551762,-3.855337,15.300000,...,11.2,10.9,0.2,0.500000,0.2,10.900000,11.058333,1.333333,0,1.000000
93,370400000,0,5,0.014107,0.183826,23.906983,16.787835,1.731028,15.056807,20.500000,...,12.0,10.6,5.2,3.176471,0.2,10.363043,10.733333,2.350000,0,1.000000
94,368144120,0,6,0.003999,4.377579,10.309683,15.735415,4.378995,11.356420,40.666667,...,22.5,0.0,-21.6,58.129412,141.9,19.266667,0.008824,1.516667,0,0.333333
95,367642410,0,7,0.003019,3.120018,0.656983,3.122940,3.122245,0.000695,56.683333,...,0.1,0.0,0.0,4.266667,7.5,0.000000,0.004167,49.833333,1,0.000000
96,367007750,0,8,0.001725,3.345271,16.473650,4.004685,4.040150,-0.035465,58.816667,...,5.0,0.2,0.0,38.388000,132.3,0.666667,2.066667,1.833333,0,0.320000
97,367453420,0,9,0.001143,2.158515,29.406983,10.167515,2.158515,8.009000,10.816667,...,9.8,7.7,-2.1,5.394000,0.3,7.482000,7.700000,49.800000,1,1.000000
98,367357150,0,10,0.000752,3.970684,15.026350,4.039066,3.998695,0.040372,58.316667,...,1.9,0.2,0.0,16.800000,51.0,0.523077,0.044444,1.816667,0,0.125000



EVENT: V3_EVENT_0176


,MMSI,label,model_rank,predicted_probability,closest_distance_km,closest_time_difference_min,distance_at_window_start_km,distance_at_window_end_km,distance_change_km,time_inside_source_radius_min,...,maximum_sog,speed_at_closest_approach,speed_change,average_course_change,course_change_at_closest_approach,approach_speed,departure_speed,maximum_ais_gap_min,long_ais_gap_count,moving_fraction
99,352986193,0,1,0.713766,1.245117,5.338748,8.320005,6.627663,1.692342,40.350000,...,10.5,6.3,1.1,3.390000,1.8,8.273333,8.766667,1.850000,0,1.0
100,366957490,1,2,0.428292,1.045487,2.555415,11.480866,5.127138,6.353727,31.666667,...,24.2,20.0,4.0,23.602041,7.7,14.446429,12.036364,2.300000,0,1.0
101,563656000,0,3,0.078067,1.406787,18.738748,7.740953,3.309871,4.431082,38.833333,...,8.6,8.5,3.6,5.765306,4.4,6.124390,8.000000,1.833333,0,1.0
102,357659000,0,4,0.026310,3.318545,13.705415,14.296383,7.708196,6.588187,20.816667,...,15.2,15.1,-0.6,3.436538,7.0,12.350000,14.940000,1.500000,0,1.0
103,636020148,0,5,0.005460,2.442938,6.438748,11.356561,7.730224,3.626337,25.350000,...,12.5,12.1,2.2,3.095833,4.0,10.563333,12.257895,2.150000,0,1.0
104,311050800,0,6,0.002740,2.401700,25.288748,20.191868,3.840960,16.350908,12.500000,...,20.7,20.6,6.2,6.404082,4.1,14.464444,20.560000,1506.366667,1,1.0
105,972971214,0,7,0.002218,2.453042,24.738748,19.806193,3.775640,16.030553,8.966667,...,22.8,18.9,5.8,7.526316,3.1,15.220000,20.460000,27.700000,0,1.0



EVENT: V3_EVENT_0212


,MMSI,label,model_rank,predicted_probability,closest_distance_km,closest_time_difference_min,distance_at_window_start_km,distance_at_window_end_km,distance_change_km,time_inside_source_radius_min,...,maximum_sog,speed_at_closest_approach,speed_change,average_course_change,course_change_at_closest_approach,approach_speed,departure_speed,maximum_ais_gap_min,long_ais_gap_count,moving_fraction
116,367640910,0,1,0.220835,0.904989,8.498494,0.904989,21.784239,-20.879250,8.600000,...,19.4,18.1,1.1,3.985714,0.0,18.100000,18.808571,1.416667,0,1.0
117,367458710,1,2,0.076415,0.884948,14.118173,1.159843,4.389057,-3.229214,17.516667,...,9.8,9.3,0.4,3.981250,4.1,9.200000,9.385714,1.500000,0,1.0
118,367739650,0,3,0.023973,2.191723,5.081827,16.165263,2.653615,13.511648,9.000000,...,21.2,20.6,-0.6,6.312000,7.9,20.678261,20.600000,1.800000,0,1.0
119,367582020,0,4,0.000335,2.678370,29.234840,2.863765,2.678370,0.185395,2.333333,...,8.5,8.4,0.1,0.500000,1.2,8.400000,8.400000,1.166667,0,1.0



EVENT: V3_EVENT_0234


,MMSI,label,model_rank,predicted_probability,closest_distance_km,closest_time_difference_min,distance_at_window_start_km,distance_at_window_end_km,distance_change_km,time_inside_source_radius_min,...,maximum_sog,speed_at_closest_approach,speed_change,average_course_change,course_change_at_closest_approach,approach_speed,departure_speed,maximum_ais_gap_min,long_ais_gap_count,moving_fraction
133,636017947,0,1,0.345544,1.101480,10.105650,14.106239,1.101480,13.004759,11.516667,...,11.9,10.5,-1.4,2.577143,2.7,10.728571,10.5,1.366667,0,1.0
134,477748900,1,2,0.344954,1.067919,0.977683,10.221955,1.067919,9.154036,10.316667,...,11.6,11.4,1.9,0.416667,0.0,10.654167,11.4,1.816667,0,1.0
135,367007750,0,3,0.009936,1.861226,25.155650,1.868824,1.872277,-0.003453,59.650000,...,0.1,0.0,0.0,7.284906,6.1,0.004082,0.0,1.350000,0,0.0
136,367357150,0,4,0.001083,2.416787,29.905650,2.419131,2.416787,0.002345,58.983333,...,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,1.650000,0,0.0


In [59]:
# V3 Feature Importance

import pandas as pd

feature_importance_v3 = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": realistic_model_v3.feature_importances_
})

feature_importance_v3 = feature_importance_v3.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

feature_importance_v3

,Feature,Importance
0,closest_distance_km,0.146088
1,moving_fraction,0.137928
2,closest_time_difference_min,0.132750
3,maximum_sog,0.060631
4,time_inside_source_radius_min,0.056282
5,average_sog,0.049880
6,distance_at_window_start_km,0.049417
7,approach_speed,0.046863
8,average_course_change,0.044930
9,speed_at_closest_approach,0.042511


In [60]:
def find_ais_candidates(
    ais_df,
    source_lat,
    source_lon,
    source_time,
    source_radius_km=5,
    time_window_minutes=30
):
    """
    Find vessels that come within the spatial and temporal
    search window around an estimated spill origin.

    Returns:
        candidate_mmsis
        candidate_points
    """

    data = ais_df.copy()

    # Make sure timestamps are datetime
    data["BaseDateTime"] = pd.to_datetime(
        data["BaseDateTime"],
        utc=True
    )

    source_time = pd.to_datetime(
        source_time,
        utc=True
    )

    # Calculate distance from estimated spill origin
    data["distance_to_source_km"] = haversine_distance(
        data["LAT"],
        data["LON"],
        source_lat,
        source_lon
    )

    # Calculate time difference from estimated spill time
    data["time_difference_min"] = (
        (data["BaseDateTime"] - source_time)
        .abs()
        .dt.total_seconds() / 60
    )

    # Apply spatial + temporal filtering
    candidate_points = data[
        (data["distance_to_source_km"] <= source_radius_km) &
        (data["time_difference_min"] <= time_window_minutes)
    ].copy()

    # Get unique candidate vessels
    candidate_mmsis = (
        candidate_points["MMSI"]
        .dropna()
        .unique()
        .tolist()
    )

    return candidate_mmsis, candidate_points

In [61]:
def rank_ais_candidates(
    ais_df,
    model,
    source_lat,
    source_lon,
    source_time,
    source_radius_km=5,
    time_window_minutes=30
):
    """
    Identify and rank potential source vessels using the trained XGBoost model.
    """

    data = ais_df.copy()

    # Ensure timestamp format
    data["BaseDateTime"] = pd.to_datetime(
        data["BaseDateTime"],
        utc=True
    )

    source_time = pd.to_datetime(
        source_time,
        utc=True
    )

    # ---------------------------------------------------------
    # 1. Find candidate vessels
    # ---------------------------------------------------------

    candidate_mmsis, candidate_points = find_ais_candidates(
        data,
        source_lat,
        source_lon,
        source_time,
        source_radius_km,
        time_window_minutes
    )

    if len(candidate_mmsis) == 0:
        return pd.DataFrame()

    # ---------------------------------------------------------
    # 2. Calculate features for each candidate
    # ---------------------------------------------------------

    candidate_feature_rows = []

    for mmsi in candidate_mmsis:

        vessel_track = data[
            (data["MMSI"] == mmsi) &
            (
                (data["BaseDateTime"] >= source_time -
                 pd.Timedelta(minutes=time_window_minutes))
            ) &
            (
                (data["BaseDateTime"] <= source_time +
                 pd.Timedelta(minutes=time_window_minutes))
            )
        ].copy()

        if len(vessel_track) == 0:
            continue

        features = calculate_candidate_features(
            vessel_track,
            source_lat,
            source_lon,
            source_time,
            source_radius_km
        )

        features["MMSI"] = mmsi

        candidate_feature_rows.append(features)

    if len(candidate_feature_rows) == 0:
        return pd.DataFrame()

    feature_df = pd.DataFrame(candidate_feature_rows)

    # ---------------------------------------------------------
    # 3. Prepare model input
    # ---------------------------------------------------------

    X = feature_df[FEATURES].copy()

    # ---------------------------------------------------------
    # 4. XGBoost prediction
    # ---------------------------------------------------------

    feature_df["source_probability"] = model.predict_proba(X)[:, 1]

    # ---------------------------------------------------------
    # 5. Rank candidates
    # ---------------------------------------------------------

    feature_df = feature_df.sort_values(
        "source_probability",
        ascending=False
    ).reset_index(drop=True)

    feature_df["rank"] = (
        feature_df.index + 1
    )

    return feature_df

In [62]:
# Select one V3 event
test_event = realistic_event_df_v3.iloc[0]

test_lat = test_event["source_lat"]
test_lon = test_event["source_lon"]
test_time = test_event["source_time"]

print("Test Event:", test_event["event_id"])
print("Estimated Latitude:", test_lat)
print("Estimated Longitude:", test_lon)
print("Estimated Time:", test_time)
print("Actual Source MMSI:", test_event["actual_source_mmsi"])

Test Event: V3_EVENT_0001
Estimated Latitude: 28.69234091010553
Estimated Longitude: -89.55970048791042
Estimated Time: 2022-01-03 08:48:22.285751537+00:00
Actual Source MMSI: 477288000


In [63]:
# Run the AIS inference pipeline on the test event

results_v3_test = rank_ais_candidates(
    ais_df=ais,
    model=realistic_model_v3,
    source_lat=test_lat,
    source_lon=test_lon,
    source_time=test_time,
    source_radius_km=5,
    time_window_minutes=30
)

print("Number of candidate vessels:", len(results_v3_test))

results_v3_test[
    [
        "rank",
        "MMSI",
        "source_probability",
        "closest_distance_km",
        "closest_time_difference_min"
    ]
]

Number of candidate vessels: 3


,rank,MMSI,source_probability,closest_distance_km,closest_time_difference_min
0,1,477288000,0.976105,0.614751,12.188096
1,2,538007823,0.018026,0.471148,23.861904
2,3,367373000,0.001587,3.504300,13.361904


In [64]:
# Test the inference pipeline on all V3 test events

test_event_ids = ranking_results_v3["event_id"].unique()

evaluation_rows = []

for event_id in test_event_ids:

    event = realistic_event_df_v3[
        realistic_event_df_v3["event_id"] == event_id
    ].iloc[0]

    # Use ONLY the estimated source information
    results = rank_ais_candidates(
        ais_df=ais,
        model=realistic_model_v3,
        source_lat=event["source_lat"],
        source_lon=event["source_lon"],
        source_time=event["source_time"],
        source_radius_km=5,
        time_window_minutes=30
    )

    actual_mmsi = event["actual_source_mmsi"]

    if results.empty:
        evaluation_rows.append({
            "event_id": event_id,
            "actual_source_mmsi": actual_mmsi,
            "predicted_rank": None,
            "candidate_count": 0,
            "found": False
        })
        continue

    matching = results[
        results["MMSI"] == actual_mmsi
    ]

    if matching.empty:
        evaluation_rows.append({
            "event_id": event_id,
            "actual_source_mmsi": actual_mmsi,
            "predicted_rank": None,
            "candidate_count": len(results),
            "found": False
        })
    else:
        source_rank = int(
            matching.iloc[0]["rank"]
        )

        evaluation_rows.append({
            "event_id": event_id,
            "actual_source_mmsi": actual_mmsi,
            "predicted_rank": source_rank,
            "candidate_count": len(results),
            "found": True
        })

inference_evaluation_v3 = pd.DataFrame(evaluation_rows)

print("Total test events:", len(inference_evaluation_v3))
print(
    "Source found in candidate set:",
    inference_evaluation_v3["found"].sum()
)

print("\nRank distribution:")
print(
    inference_evaluation_v3[
        inference_evaluation_v3["found"]
    ]["predicted_rank"].value_counts().sort_index()
)

Total test events: 60
Source found in candidate set: 60

Rank distribution:
predicted_rank
1    55
2     4
3     1
Name: count, dtype: int64


In [65]:
def prepare_dashboard_output(
    ranked_results,
    ais_df,
    source_lat,
    source_lon,
    source_time,
    top_n=5
):
    """
    Convert the internal AIS/XGBoost results into a clean
    dashboard-friendly table.
    """

    results = ranked_results.head(top_n).copy()

    # Add vessel information where available
    vessel_info = (
        ais_df[
            ["MMSI", "VesselName", "VesselType"]
        ]
        .drop_duplicates("MMSI")
    )

    results = results.merge(
        vessel_info,
        on="MMSI",
        how="left"
    )

    # Keep only fields useful for the dashboard
    dashboard_output = results[
        [
            "rank",
            "MMSI",
            "VesselName",
            "VesselType",
            "source_probability",
            "closest_distance_km",
            "closest_time_difference_min",
            "average_sog",
            "speed_at_closest_approach",
            "moving_fraction"
        ]
    ].copy()

    # Clean names / missing values
    dashboard_output["VesselName"] = (
        dashboard_output["VesselName"]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )

    # Round values for dashboard display
    dashboard_output["source_probability"] = (
        dashboard_output["source_probability"].round(4)
    )

    dashboard_output["closest_distance_km"] = (
        dashboard_output["closest_distance_km"].round(3)
    )

    dashboard_output["closest_time_difference_min"] = (
        dashboard_output["closest_time_difference_min"].round(2)
    )

    dashboard_output["average_sog"] = (
        dashboard_output["average_sog"].round(2)
    )

    dashboard_output["speed_at_closest_approach"] = (
        dashboard_output["speed_at_closest_approach"].round(2)
    )

    dashboard_output["moving_fraction"] = (
        dashboard_output["moving_fraction"].round(3)
    )

    return dashboard_output

In [ ]:
dashboard_test_output = prepare_dashboard_output(
    ranked_results=results_v3_test,
    ais_df=ais,
    source_lat=test_lat,
    source_lon=test_lon,
    source_time=test_time,
    top_n=5
)

dashboard_test_output

In [ ]:
ais["VesselType"].value_counts(dropna=False).head(20)

In [66]:
# AIS Vessel Type Code → Human-readable label

AIS_VESSEL_TYPE_MAP = {
    0: "Unknown",
    20: "Wing-in-Ground",
    30: "Fishing",
    31: "Towing",
    32: "Towing (Long/Wide Tow)",
    33: "Dredging / Underwater Ops",
    34: "Diving Operations",
    35: "Military",
    36: "Sailing",
    37: "Pleasure Craft",
    40: "High-Speed Craft",
    50: "Pilot Vessel",
    51: "Search & Rescue",
    52: "Tug",
    53: "Port Tender",
    54: "Anti-Pollution Vessel",
    55: "Law Enforcement",
    56: "Spare / Local Assignment",
    57: "Spare / Local Assignment",
    58: "Medical Transport",
    59: "Other Special Vessel",
    60: "Passenger",
    70: "Cargo",
    80: "Tanker",
    90: "Other",
    99: "Other / Unknown"
}

# Create readable vessel type
ais["VesselTypeLabel"] = (
    ais["VesselType"]
    .map(AIS_VESSEL_TYPE_MAP)
    .fillna("Unknown")
)

# Check the result
ais[["VesselType", "VesselTypeLabel"]].drop_duplicates().sort_values("VesselType")

,VesselType,VesselTypeLabel
108952,0.0,Unknown
112330,3.0,Unknown
111854,20.0,Wing-in-Ground
119463,29.0,Unknown
41921,30.0,Fishing
15295,31.0,Towing
118866,33.0,Dredging / Underwater Ops
16922,35.0,Military
56016,37.0,Pleasure Craft
115335,39.0,Unknown


In [67]:
def prepare_dashboard_output(
    ranked_results,
    ais_df,
    source_lat,
    source_lon,
    source_time,
    top_n=5
):
    """
    Convert internal AIS/XGBoost results into a clean
    dashboard-friendly table.
    """

    results = ranked_results.head(top_n).copy()

    # Vessel information
    vessel_info = (
        ais_df[["MMSI", "VesselName", "VesselTypeLabel"]]
        .drop_duplicates("MMSI")
    )

    results = results.merge(
        vessel_info,
        on="MMSI",
        how="left"
    )

    dashboard_output = results[
        [
            "rank",
            "MMSI",
            "VesselName",
            "VesselTypeLabel",
            "source_probability",
            "closest_distance_km",
            "closest_time_difference_min",
            "average_sog",
            "speed_at_closest_approach",
            "moving_fraction"
        ]
    ].copy()

    # Clean dashboard values
    dashboard_output["VesselName"] = (
        dashboard_output["VesselName"]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )

    dashboard_output["VesselTypeLabel"] = (
        dashboard_output["VesselTypeLabel"]
        .fillna("Unknown")
    )

    dashboard_output["source_probability"] = (
        dashboard_output["source_probability"].round(4)
    )

    dashboard_output["closest_distance_km"] = (
        dashboard_output["closest_distance_km"].round(3)
    )

    dashboard_output["closest_time_difference_min"] = (
        dashboard_output["closest_time_difference_min"].round(2)
    )

    dashboard_output["average_sog"] = (
        dashboard_output["average_sog"].round(2)
    )

    dashboard_output["speed_at_closest_approach"] = (
        dashboard_output["speed_at_closest_approach"].round(2)
    )

    dashboard_output["moving_fraction"] = (
        dashboard_output["moving_fraction"].round(3)
    )

    return dashboard_output

In [68]:
dashboard_test_output = prepare_dashboard_output(
    ranked_results=results_v3_test,
    ais_df=ais,
    source_lat=test_lat,
    source_lon=test_lon,
    source_time=test_time,
    top_n=5
)

dashboard_test_output

,rank,MMSI,VesselName,VesselTypeLabel,source_probability,closest_distance_km,closest_time_difference_min,average_sog,speed_at_closest_approach,moving_fraction
0,1,477288000,OCEAN PRINCESS,Cargo,0.9761,0.615,12.19,13.87,13.6,1.0
1,2,538007823,LESEDI QUEEN,Cargo,0.0180,0.471,23.86,12.27,12.5,1.0
2,3,367373000,TRUE NORTH,Pleasure Craft,0.0016,3.504,13.36,0.00,0.0,0.0


In [69]:
# ============================================
# SAVE FINAL V3 AIS MODEL
# ============================================

import os
import json

# Create a folder for the final AIS model
MODEL_DIR = "AIS_MODEL"
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. Save the trained XGBoost model
model_path = os.path.join(MODEL_DIR, "xgboost_v3.json")
realistic_model_v3.save_model(model_path)

# 2. Save the exact feature list used by the model
features_path = os.path.join(MODEL_DIR, "features.json")

with open(features_path, "w") as f:
    json.dump(FEATURES, f, indent=4)

print("Model saved to:", model_path)
print("Features saved to:", features_path)

print("\nFiles created:")
print(os.listdir(MODEL_DIR))

Model saved to: AIS_MODEL/xgboost_v3.json
Features saved to: AIS_MODEL/features.json

Files created:
['xgboost_v3.json', 'features.json']


In [70]:
# ============================================
# VERIFY SAVED AIS MODEL
# ============================================

import xgboost as xgb
import json

# Load the saved model
loaded_model = xgb.XGBClassifier()
loaded_model.load_model("AIS_MODEL/xgboost_v3.json")

# Load the saved feature list
with open("AIS_MODEL/features.json", "r") as f:
    loaded_features = json.load(f)

print("Model loaded successfully.")
print("Number of features:", len(loaded_features))
print("Features:", loaded_features)

# Check that the saved features match our original features
print("\nFeature list matches:",
      loaded_features == FEATURES)

Model loaded successfully.
Number of features: 18
Features: ['closest_distance_km', 'distance_at_window_start_km', 'distance_at_window_end_km', 'distance_change_km', 'closest_time_difference_min', 'time_inside_source_radius_min', 'ais_point_count', 'average_sog', 'maximum_sog', 'speed_at_closest_approach', 'speed_change', 'average_course_change', 'course_change_at_closest_approach', 'approach_speed', 'departure_speed', 'maximum_ais_gap_min', 'long_ais_gap_count', 'moving_fraction']

Feature list matches: True


In [71]:
# ============================================
# VERIFY ORIGINAL VS SAVED MODEL
# ============================================

# Use the first unseen V3 test event
test_event_id = ranking_results_v3["event_id"].unique()[0]

event_data = realistic_event_df_v3[
    realistic_event_df_v3["event_id"] == test_event_id
].iloc[0]

source_lat = event_data["source_lat"]
source_lon = event_data["source_lon"]
source_time = event_data["source_time"]

print("Testing event:", test_event_id)
print("Source latitude:", source_lat)
print("Source longitude:", source_lon)
print("Source time:", source_time)

# Rank using the original model
original_result = rank_ais_candidates(
    ais,
    realistic_model_v3,
    source_lat,
    source_lon,
    source_time
)

# Rank using the saved-and-loaded model
loaded_result = rank_ais_candidates(
    ais,
    loaded_model,
    source_lat,
    source_lon,
    source_time
)

print("\nOriginal model result:")
print(
    original_result[
        ["rank", "MMSI", "source_probability"]
    ]
)

print("\nLoaded model result:")
print(
    loaded_result[
        ["rank", "MMSI", "source_probability"]
    ]
)

# Compare the important outputs
same_mmsi_order = (
    original_result["MMSI"].tolist()
    == loaded_result["MMSI"].tolist()
)

same_scores = np.allclose(
    original_result["source_probability"].values,
    loaded_result["source_probability"].values
)

print("\nMMSI ranking identical:", same_mmsi_order)
print("Scores identical:", same_scores)

Testing event: V3_EVENT_0006
Source latitude: 28.845281732765603
Source longitude: -89.47598474469574
Source time: 2022-01-04 22:51:09.636037328+00:00

Original model result:
   rank       MMSI  source_probability
0     1  367582190            0.886187
1     2  367590950            0.342683
2     3  538004541            0.004543
3     4  367603350            0.001900

Loaded model result:
   rank       MMSI  source_probability
0     1  367582190            0.886187
1     2  367590950            0.342683
2     3  538004541            0.004543
3     4  367603350            0.001900

MMSI ranking identical: True
Scores identical: True


In [86]:
def identify_source_vessels(
    source_lat,
    source_lon,
    source_time,
    ais_df,
    top_n=5,
    source_radius_km=5,
    time_window_minutes=30
):
    """
    Dynamically identify and rank potential source vessels
    using AIS data and the saved XGBoost model.

    Parameters
    ----------
    source_lat : float
        Estimated spill source latitude.

    source_lon : float
        Estimated spill source longitude.

    source_time : datetime
        Estimated spill source time.

    ais_df : pandas.DataFrame
        AIS dataset to use.

    top_n : int
        Number of vessels returned.

    source_radius_km : float
        Spatial candidate search radius.

    time_window_minutes : int
        Temporal candidate search window.
    """

    # --------------------------------------------------------
    # 1. Find and rank AIS candidates
    # --------------------------------------------------------

    ranked_results = rank_ais_candidates(
        ais_df=ais_df,
        model=loaded_model,
        source_lat=source_lat,
        source_lon=source_lon,
        source_time=source_time,
        source_radius_km=source_radius_km,
        time_window_minutes=time_window_minutes
    )

    # --------------------------------------------------------
    # 2. No candidates
    # --------------------------------------------------------

    if ranked_results.empty:

        return {
            "source_latitude": float(source_lat),
            "source_longitude": float(source_lon),
            "source_time": str(source_time),
            "candidate_vessels": []
        }

    # --------------------------------------------------------
    # 3. Prepare dashboard output
    # --------------------------------------------------------

    dashboard_output = prepare_dashboard_output(
        ranked_results=ranked_results,
        ais_df=ais_df,
        source_lat=source_lat,
        source_lon=source_lon,
        source_time=source_time,
        top_n=top_n
    )

    # --------------------------------------------------------
    # 4. Rename model probability
    # --------------------------------------------------------

    dashboard_output = dashboard_output.rename(
        columns={
            "source_probability":
                "source_consistency_score"
        }
    )

    # --------------------------------------------------------
    # 5. Convert to dashboard-friendly records
    # --------------------------------------------------------

    candidate_vessels = dashboard_output.to_dict(
        orient="records"
    )

    # --------------------------------------------------------
    # 6. Return complete result
    # --------------------------------------------------------

    return {
        "source_latitude": float(source_lat),
        "source_longitude": float(source_lon),
        "source_time": str(source_time),
        "candidate_vessels": candidate_vessels
    }

In [73]:
# ============================================
# TEST FINAL AIS INTEGRATION FUNCTION
# ============================================

# Pick one existing unseen V3 test event
test_event_id = ranking_results_v3["event_id"].unique()[0]

event_data = realistic_event_df_v3[
    realistic_event_df_v3["event_id"] == test_event_id
].iloc[0]

# Get the estimated source information
test_lat = event_data["source_lat"]
test_lon = event_data["source_lon"]
test_time = event_data["source_time"]

print("Testing event:", test_event_id)
print("Estimated source:")
print("Latitude :", test_lat)
print("Longitude:", test_lon)
print("Time     :", test_time)

# Run the FINAL integration function
ais_result = identify_source_vessels(
    source_lat=test_lat,
    source_lon=test_lon,
    source_time=test_time,
    top_n=5
)

print("\nAIS INTEGRATION OUTPUT:")
print(ais_result)

Testing event: V3_EVENT_0006
Estimated source:
Latitude : 28.845281732765603
Longitude: -89.47598474469574
Time     : 2022-01-04 22:51:09.636037328+00:00

AIS INTEGRATION OUTPUT:
{'source_latitude': 28.845281732765603, 'source_longitude': -89.47598474469574, 'source_time': '2022-01-04 22:51:09.636037328+00:00', 'candidate_vessels': [{'rank': 1, 'MMSI': 367582190, 'VesselName': 'LYLA ANGELLE', 'VesselTypeLabel': 'Passenger', 'source_consistency_score': 0.8862000107765198, 'closest_distance_km': 0.531, 'closest_time_difference_min': 5.87, 'average_sog': 13.79, 'speed_at_closest_approach': 16.9, 'moving_fraction': 0.956}, {'rank': 2, 'MMSI': 367590950, 'VesselName': 'DIAMOND MINE', 'VesselTypeLabel': 'Passenger', 'source_consistency_score': 0.3427000045776367, 'closest_distance_km': 0.859, 'closest_time_difference_min': 25.64, 'average_sog': 4.2, 'speed_at_closest_approach': 16.1, 'moving_fraction': 0.844}, {'rank': 3, 'MMSI': 538004541, 'VesselName': 'IONIC HUNTRESS', 'VesselTypeLabel': 

In [74]:
print("=" * 60)
print("        AIS VESSEL IDENTIFICATION RESULTS")
print("=" * 60)

print("\nMODEL EVALUATION")
print("-" * 60)
print("Training Events       : 300")
print("Unseen Test Events    : 60")
print("Test Candidate Rows   : 188")

print("\nCLASSIFICATION METRICS")
print("-" * 60)
print("Accuracy              : 86.70%")
print("Precision             : 81.82%")
print("Recall                : 75.00%")
print("F1 Score              : 78.26%")
print("ROC-AUC               : 0.9473")

print("\nRANKING PERFORMANCE")
print("-" * 60)
print("Top-1                 : 91.67%")
print("Top-3                 : 100.00%")
print("Top-5                 : 100.00%")

print("\nMULTI-CANDIDATE EVENTS")
print("-" * 60)
print("Test Events           : 41")
print("Top-1                 : 87.80%")
print("Top-3                 : 100.00%")

print("\n" + "=" * 60)

        AIS VESSEL IDENTIFICATION RESULTS

MODEL EVALUATION
------------------------------------------------------------
Training Events       : 300
Unseen Test Events    : 60
Test Candidate Rows   : 188

CLASSIFICATION METRICS
------------------------------------------------------------
Accuracy              : 86.70%
Precision             : 81.82%
Recall                : 75.00%
F1 Score              : 78.26%
ROC-AUC               : 0.9473

RANKING PERFORMANCE
------------------------------------------------------------
Top-1                 : 91.67%
Top-3                 : 100.00%
Top-5                 : 100.00%

MULTI-CANDIDATE EVENTS
------------------------------------------------------------
Test Events           : 41
Top-1                 : 87.80%
Top-3                 : 100.00%



In [80]:
# ============================================================
# SYNTHETIC CHENNAI AIS — INTEGRATION BRANCH
# ============================================================

ais_chennai = chennai_ais.copy()

# Convert timestamp to UTC
ais_chennai["BaseDateTime"] = pd.to_datetime(
    ais_chennai["BaseDateTime"],
    utc=True
)

# Sort by vessel and time
ais_chennai = ais_chennai.sort_values(
    ["MMSI", "BaseDateTime"]
).reset_index(drop=True)

print("Synthetic AIS prepared.")
print("Rows:", len(ais_chennai))
print("Vessels:", ais_chennai["MMSI"].nunique())
print(
    "Time range:",
    ais_chennai["BaseDateTime"].min(),
    "to",
    ais_chennai["BaseDateTime"].max()
)

Synthetic AIS prepared.
Rows: 1225
Vessels: 25
Time range: 2026-09-08 04:00:00+00:00 to 2026-09-08 08:00:00+00:00


In [81]:
# ============================================================
# SYNTHETIC AIS — FEATURE ENGINEERING
# ============================================================

# Work on a copy
ais_chennai = ais_chennai.copy()

# ------------------------------------------------------------
# 1. Previous position and time
# ------------------------------------------------------------

ais_chennai["previous_LAT"] = (
    ais_chennai.groupby("MMSI")["LAT"].shift(1)
)

ais_chennai["previous_LON"] = (
    ais_chennai.groupby("MMSI")["LON"].shift(1)
)

ais_chennai["previous_time"] = (
    ais_chennai.groupby("MMSI")["BaseDateTime"].shift(1)
)

# ------------------------------------------------------------
# 2. Time difference
# ------------------------------------------------------------

ais_chennai["time_diff_seconds"] = (
    ais_chennai["BaseDateTime"] -
    ais_chennai["previous_time"]
).dt.total_seconds()

# ------------------------------------------------------------
# 3. Distance from previous AIS point
# ------------------------------------------------------------

ais_chennai["distance_from_previous_km"] = haversine_distance(
    ais_chennai["previous_LAT"],
    ais_chennai["previous_LON"],
    ais_chennai["LAT"],
    ais_chennai["LON"]
)

# First point of each vessel has no previous position
ais_chennai["distance_from_previous_km"] = (
    ais_chennai["distance_from_previous_km"]
    .fillna(0)
)

# ------------------------------------------------------------
# 4. Calculated speed
# ------------------------------------------------------------

ais_chennai["calculated_speed_kmh"] = np.where(
    ais_chennai["time_diff_seconds"] > 0,
    (
        ais_chennai["distance_from_previous_km"] /
        ais_chennai["time_diff_seconds"]
    ) * 3600,
    0
)

ais_chennai["calculated_speed_knots"] = (
    ais_chennai["calculated_speed_kmh"] / 1.852
)

# ------------------------------------------------------------
# 5. Speed difference
# ------------------------------------------------------------

ais_chennai["speed_difference_knots"] = (
    ais_chennai.groupby("MMSI")["SOG"].diff()
)

ais_chennai["speed_difference_knots"] = (
    ais_chennai["speed_difference_knots"].fillna(0)
)

# ------------------------------------------------------------
# 6. Previous course
# ------------------------------------------------------------

ais_chennai["previous_COG"] = (
    ais_chennai.groupby("MMSI")["COG"].shift(1)
)

# ------------------------------------------------------------
# 7. Course change
# ------------------------------------------------------------

course_difference = (
    ais_chennai["COG"] -
    ais_chennai["previous_COG"]
).abs()

ais_chennai["course_change_deg"] = np.minimum(
    course_difference,
    360 - course_difference
)

ais_chennai["course_change_deg"] = (
    ais_chennai["course_change_deg"].fillna(0)
)

# ------------------------------------------------------------
# 8. AIS time gaps
# ------------------------------------------------------------

ais_chennai["ais_gap_minutes"] = (
    ais_chennai["time_diff_seconds"] / 60
)

ais_chennai["ais_gap_minutes"] = (
    ais_chennai["ais_gap_minutes"].fillna(0)
)

ais_chennai["is_long_ais_gap"] = (
    ais_chennai["ais_gap_minutes"] > 30
)

# ------------------------------------------------------------
# 9. Vessel type labels
# ------------------------------------------------------------

AIS_VESSEL_TYPE_MAP = {
    0: "Unknown",
    20: "Wing-in-Ground",
    30: "Fishing",
    31: "Towing",
    32: "Towing (Long/Wide Tow)",
    33: "Dredging / Underwater Ops",
    34: "Diving Operations",
    35: "Military",
    36: "Sailing",
    37: "Pleasure Craft",
    40: "High-Speed Craft",
    50: "Pilot Vessel",
    51: "Search & Rescue",
    52: "Tug",
    53: "Port Tender",
    54: "Anti-Pollution Vessel",
    55: "Law Enforcement",
    56: "Spare / Local Assignment",
    57: "Spare / Local Assignment",
    58: "Medical Transport",
    59: "Other Special Vessel",
    60: "Passenger",
    70: "Cargo",
    80: "Tanker",
    90: "Other",
    99: "Other / Unknown"
}

ais_chennai["VesselTypeLabel"] = (
    ais_chennai["VesselType"]
    .map(AIS_VESSEL_TYPE_MAP)
    .fillna("Unknown")
)

# ------------------------------------------------------------
# 10. Sort again
# ------------------------------------------------------------

ais_chennai = ais_chennai.sort_values(
    ["MMSI", "BaseDateTime"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 11. Check result
# ------------------------------------------------------------

print("==========================================")
print("SYNTHETIC AIS FEATURE ENGINEERING")
print("==========================================")

print(f"Rows: {len(ais_chennai):,}")
print(f"Vessels: {ais_chennai['MMSI'].nunique()}")

print("\nNew feature columns:")

for column in [
    "time_diff_seconds",
    "distance_from_previous_km",
    "calculated_speed_knots",
    "speed_difference_knots",
    "course_change_deg",
    "ais_gap_minutes",
    "is_long_ais_gap",
    "VesselTypeLabel"
]:
    print(f"  ✓ {column}")

print("\nMissing values in navigation features:")

print(
    ais_chennai[
        [
            "time_diff_seconds",
            "distance_from_previous_km",
            "calculated_speed_knots",
            "speed_difference_knots",
            "course_change_deg",
            "ais_gap_minutes"
        ]
    ].isna().sum()
)

SYNTHETIC AIS FEATURE ENGINEERING
Rows: 1,225
Vessels: 25

New feature columns:
  ✓ time_diff_seconds
  ✓ distance_from_previous_km
  ✓ calculated_speed_knots
  ✓ speed_difference_knots
  ✓ course_change_deg
  ✓ ais_gap_minutes
  ✓ is_long_ais_gap
  ✓ VesselTypeLabel

Missing values in navigation features:
time_diff_seconds            25
distance_from_previous_km     0
calculated_speed_knots        0
speed_difference_knots        0
course_change_deg             0
ais_gap_minutes               0
dtype: int64


In [82]:
# ============================================================
# 18-FEATURE EXTRACTION TEST — CORRECT LAGRANGIAN SOURCE
# ============================================================

# These values come from the Lagrangian model output
TEST_SOURCE_LAT = 12.22076
TEST_SOURCE_LON = 80.13420
TEST_SOURCE_TIME = pd.Timestamp(
    "2026-09-08 06:00:00",
    tz="UTC"
)

# Pick the controlled synthetic vessel
TEST_MMSI = ais_chennai["MMSI"].iloc[0]

test_vessel_track = ais_chennai[
    ais_chennai["MMSI"] == TEST_MMSI
].copy()

test_features = calculate_candidate_features(
    test_vessel_track,
    TEST_SOURCE_LAT,
    TEST_SOURCE_LON,
    TEST_SOURCE_TIME,
    source_radius_km=5
)

print("==========================================")
print("18-FEATURE EXTRACTION TEST")
print("==========================================")

print(f"MMSI: {TEST_MMSI}")
print(f"Vessel: {test_vessel_track['VesselName'].iloc[0]}")

print("\nSource used:")
print(f"Latitude  : {TEST_SOURCE_LAT}")
print(f"Longitude : {TEST_SOURCE_LON}")
print(f"Time      : {TEST_SOURCE_TIME}")

print("\nNumber of features:")
print(len(test_features))

print("\nFeatures:")

for feature in FEATURES:
    print(
        f"{feature:35s}: "
        f"{test_features[feature]}"
    )

print("\nMissing values:")

test_feature_series = pd.Series(test_features)

print(
    test_feature_series[FEATURES]
    .isna()
    .sum()
)

18-FEATURE EXTRACTION TEST
MMSI: 419000000
Vessel: CHENNAI TRADER

Source used:
Latitude  : 12.22076
Longitude : 80.1342
Time      : 2026-09-08 06:00:00+00:00

Number of features:
18

Features:
closest_distance_km                : 0.0
distance_at_window_start_km        : 37.01342656472001
distance_at_window_end_km          : 37.01372580768249
distance_change_km                 : -0.00029924296248395876
closest_time_difference_min        : 0.0
time_inside_source_radius_min      : 30.0
ais_point_count                    : 49
average_sog                        : 10.0
maximum_sog                        : 10.0
speed_at_closest_approach          : 10.0
speed_change                       : 0.0
average_course_change              : 0.0
course_change_at_closest_approach  : 0.0
approach_speed                     : 10.0
departure_speed                    : 10.0
maximum_ais_gap_min                : 5.0
long_ais_gap_count                 : 0
moving_fraction                    : 1.0

Missing values:


In [83]:
# ============================================================
# LAGRANGIAN → SYNTHETIC AIS → XGBOOST INTEGRATION TEST
# ============================================================

# Temporarily use the synthetic Chennai AIS dataset
ais_integration_test = ais_chennai.copy()

# ------------------------------------------------------------
# Find candidate vessels near the Lagrangian source
# ------------------------------------------------------------

candidate_mmsis, candidate_points = find_ais_candidates(
    ais_df=ais_integration_test,
    source_lat=TEST_SOURCE_LAT,
    source_lon=TEST_SOURCE_LON,
    source_time=TEST_SOURCE_TIME,
    source_radius_km=5,
    time_window_minutes=30
)

print("==========================================")
print("AIS CANDIDATE FILTER")
print("==========================================")

print(f"Source latitude  : {TEST_SOURCE_LAT}")
print(f"Source longitude : {TEST_SOURCE_LON}")
print(f"Source time      : {TEST_SOURCE_TIME}")

print(f"\nCandidate vessels found: {len(candidate_mmsis)}")

print("\nCandidate MMSIs:")

for mmsi in candidate_mmsis:
    vessel_name = (
        ais_integration_test[
            ais_integration_test["MMSI"] == mmsi
        ]["VesselName"]
        .iloc[0]
    )

    print(f"  {mmsi} → {vessel_name}")

AIS CANDIDATE FILTER
Source latitude  : 12.22076
Source longitude : 80.1342
Source time      : 2026-09-08 06:00:00+00:00

Candidate vessels found: 1

Candidate MMSIs:
  419000000 → CHENNAI TRADER


In [84]:
# ============================================================
# XGBOOST SOURCE VESSEL RANKING TEST
# ============================================================

ranked_chennai = rank_ais_candidates(
    ais_df=ais_chennai,
    model=loaded_model,
    source_lat=TEST_SOURCE_LAT,
    source_lon=TEST_SOURCE_LON,
    source_time=TEST_SOURCE_TIME,
    source_radius_km=5,
    time_window_minutes=30
)

print("==========================================")
print("XGBOOST SOURCE VESSEL RANKING")
print("==========================================")

if ranked_chennai.empty:

    print("No candidate vessels found.")

else:

    print(f"Candidate vessels ranked: {len(ranked_chennai)}")
    print()

    print(
        ranked_chennai[
            [
                "rank",
                "MMSI",
                "source_probability",
                "closest_distance_km",
                "closest_time_difference_min",
                "average_sog",
                "speed_at_closest_approach",
                "moving_fraction"
            ]
        ].to_string(index=False)
    )

XGBOOST SOURCE VESSEL RANKING
Candidate vessels ranked: 1

 rank      MMSI  source_probability  closest_distance_km  closest_time_difference_min  average_sog  speed_at_closest_approach  moving_fraction
    1 419000000            0.928382                  0.0                          0.0         10.0                       10.0              1.0


In [79]:
# ============================================================
# ALIGN CONTROLLED VESSEL WITH LAGRANGIAN SOURCE TIME
# ============================================================

SOURCE_LAT = 12.22076
SOURCE_LON = 80.13420
SOURCE_TIME = pd.Timestamp(
    "2026-09-08 06:00:00",
    tz="UTC"
)

CONTROLLED_MMSI = 419000000

# Work on raw synthetic AIS
chennai_ais = chennai_ais.copy()

# Find the controlled vessel's position at the source time
controlled_at_source_time = chennai_ais[
    (chennai_ais["MMSI"] == CONTROLLED_MMSI) &
    (chennai_ais["BaseDateTime"] == SOURCE_TIME)
]

print("Controlled vessel records at source time:")
print(controlled_at_source_time[["MMSI", "BaseDateTime", "LAT", "LON"]])

# ------------------------------------------------------------
# Shift the entire controlled vessel track so that its
# position at 06:00 UTC is exactly the Lagrangian source.
# ------------------------------------------------------------

if len(controlled_at_source_time) == 1:

    current_lat = controlled_at_source_time.iloc[0]["LAT"]
    current_lon = controlled_at_source_time.iloc[0]["LON"]

    lat_shift = SOURCE_LAT - current_lat
    lon_shift = SOURCE_LON - current_lon

    mask = chennai_ais["MMSI"] == CONTROLLED_MMSI

    chennai_ais.loc[mask, "LAT"] += lat_shift
    chennai_ais.loc[mask, "LON"] += lon_shift

    print("\nControlled vessel trajectory aligned.")
    print(f"Latitude shift  : {lat_shift:.6f}")
    print(f"Longitude shift : {lon_shift:.6f}")

else:
    print(
        "\nERROR: Could not find exactly one "
        "controlled-vessel record at source time."
    )

Controlled vessel records at source time:
         MMSI              BaseDateTime        LAT        LON
24  419000000 2026-09-08 06:00:00+00:00  12.440842  80.365053

Controlled vessel trajectory aligned.
Latitude shift  : -0.220082
Longitude shift : -0.230853
